In [1]:
if True:
    import pandas as pd
    import sys, os

    x = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")
    if len(x) <= 10:
        !cp ../input/jigsaw-agile-community-rules/sample_submission.csv ./submission.csv

        sys.exit(0)

SystemExit: 0

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# TTT

## qwen_utils.py

In [ ]:
%%writefile qwen_utils.py

import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split

from qwen_constants import *


def save_obj(obj, filename, verbose=True):
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)

    if verbose:
        print(f'Saved: {filename}')

    return None

def load_obj(filename, verbose=True):
    with open(filename, 'rb') as f:
        obj = pickle.load(f)

    if verbose:
        print(f'Loaded: {filename}')

    return obj


def create_clf_dataset(df):
    dfs_to_concat_v = []
    if ('positive_example_1' in df.columns) and ('positive_example_2' in df.columns):
        all_positives_df = pd.concat(
            [
                df[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'}),
                df[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_positives_df['rule_violation'] = 1
        dfs_to_concat_v.append(all_positives_df)
    
    
    if ('negative_example_1' in df.columns) and ('negative_example_2' in df.columns):
        all_negatives_df = pd.concat(
            [
                df[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'}),
                df[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_negatives_df['rule_violation'] = 0
        dfs_to_concat_v.append(all_negatives_df)
    
    
    if ('body' in df.columns) and ('rule_violation' in df.columns):
        all_bodies_df = df[['rule', 'body', 'rule_violation']].copy()
        dfs_to_concat_v.append(all_bodies_df)
        
        
    if len(dfs_to_concat_v) > 1:
        all_examples_df = pd.concat(
            dfs_to_concat_v,
            axis=0
        ).drop_duplicates()
        
    elif len(dfs_to_concat_v) == 1:
        all_examples_df = dfs_to_concat_v[0].drop_duplicates()
        
    else:
        raise Exception('No data found')
        
    return all_examples_df.reset_index(drop=True)


def load_trn_val_fit_df(ds_seed):
    # Train dataset and val dataset
    df_train = pd.read_csv("../input/jigsaw-agile-community-rules/train.csv").sample(frac=PUBLIC_TRN_DS_PROP, random_state=ds_seed).reset_index(drop=True)
    
    if IS_DEBUG:
        df_tst = pd.read_csv("../input/jigsaw-agile-community-rules/train.csv").sample(frac=0.02, random_state=ds_seed).reset_index(drop=True)
    else:
        df_tst = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")

    all_train_examples_df = create_clf_dataset(df_train)[['rule', 'body', 'rule_violation']]
    all_test_examples_df = create_clf_dataset(df_tst)[['rule', 'body', 'rule_violation']]
    df_body = pd.concat(
        [
            all_test_examples_df,
            all_train_examples_df
        ], 
        axis=0
    )
    df_body = df_body.drop_duplicates(subset=['rule', 'body','rule_violation']).copy()

    print('Total samples:', len(df_body))
    print('Duplicated bodies:', df_body['body'].shape[0] - df_body['body'].nunique())

    if VAL_ON_TRAIN_DATA:
        df_trn = df_body.sample(frac=1, random_state=ds_seed+234).reset_index(drop=True)
        if VAL_DS_SIZE == 0:
            df_val = None
            
        else:
            df_val = train_test_split(
                df_body,
                test_size=VAL_DS_SIZE,
                random_state=ds_seed,
                shuffle=True,
                stratify=df_body.rule,
            )[1]

    else:
        df_trn, df_val = train_test_split(
            df_body,
            test_size=VAL_DS_SIZE,
            random_state=ds_seed,
            shuffle=True,
            stratify=df_body.rule,
        )

    if df_val is not None:
        df_val = df_val.reset_index(drop=True).copy()

    df_trn = df_trn.drop_duplicates(subset=['rule', 'body']).reset_index(drop=True).copy()
    
    if TRAIN_DS_PROP != 1:    
        df_trn = train_test_split(
            df_trn,
            train_size=TRAIN_DS_PROP,
            random_state=ds_seed+123,
            shuffle=True,
            stratify=df_trn.rule,
        )[0].reset_index(drop=True).copy()
    
    if FIT_ON_VAL:
        assert df_val is not None
        df_fit = df_val
        
    else:
        df_fit = df_trn
        
    return df_trn, df_val, df_fit



if __name__ == '__main__':
    df_trn, df_val, df_fit = load_trn_val_fit_df(DS_SEED)

    print()
    print('------ CFG ----------')
    print(f'{MODEL_NAME_V=}')
    print()
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{IS_DEBUG=} {"☢️☢️☢️☢️" if IS_DEBUG else ""}')
    print(f'{GRAD_ACC_NUM=}')
    print(f'{MAX_TOKEN_LEN=}')
    print()
    print(f'{len(df_trn)=}')
    print(f'{len(df_fit)=}')
    if df_val is None:
        print(f'{df_val=}')
    else:
        print(f'{len(df_val)=}')
    print('----------------')
    print()

## qwen_train.py

In [ ]:
%%writefile qwen_train.py


import warnings
import random

from typing import Union, Any
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

import torch
from qwen_utils import *

from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from transformers.utils import is_torch_bf16_gpu_available
from torch.utils.data import Dataset

if UNSLOTH_TRAINING:
    from unsloth import FastLanguageModel
    import torch
    
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig, TaskType, get_peft_model
    import peft
        
if __name__ != '__main__':
    print(f'⛑️⛑️⛑️ Train import for device: {os.environ.get("CUDA_VISIBLE_DEVICES")} ⛑️⛑️⛑️')


class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """
    Data collator used for completion tasks. It ensures that all the tokens of the labels are set to an 'ignore_index'
    when they do not come from the assistant. This ensure that the loss is only calculated on the completion made by
    the assistant.

    Args:
        response_template (`Union[str, list[int]]`):
            the template form that indicates the start of the response, typically something like '### Response:\n'. It
            can also be passed as tokenized ids, which can be useful when using a tokenizer that encodes the response
            differently if it does not have proper context.
        mlm (`bool`, *optional*, defaults to `False`): Whether to use masked language modeling in the underlying
            `DataCollatorForLanguageModeling` class. Note that this option currently has no effect but is present
             for flexibility and backwards-compatibility.
        ignore_index (`int`, *optional*, defaults to `-100`):
            The index to use to ignore the initial tokens with
    """

    def __init__(
        self,
        response_template: Union[str, list[int]],
        *args,
        mlm: bool = False,
        ignore_index: int = -100,
        multiple_responses: bool = False,
        resposen_num_tokens: int = 1,
        **kwargs,
    ):
        super().__init__(*args, mlm=mlm, **kwargs)


        self.response_template = response_template
        if isinstance(response_template, str):
            # The user provides a string, must tokenize
            self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        else:
            # The user already provides the token ids
            self.response_token_ids = response_template

        self.ignore_index = ignore_index
        self.multiple_responses = multiple_responses
        self.resposen_num_tokens = resposen_num_tokens
        
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)

        for i in range(len(examples)):
            response_token_ids_start_idx = None
            response_token_ids_last_idx = 0
            response_token_ids_end_idx = None
            
            for idx in torch.argwhere(batch["labels"][i] == self.response_token_ids[0]).T[0]:
                # `response_token_ids` is `'### Response:\n'`, here we are just making sure that the token IDs match
                if (
                    self.response_token_ids == batch["labels"][i, idx : idx + len(self.response_token_ids)].tolist()
                ):
                    response_token_ids_start_idx = idx
                    response_token_ids_end_idx = response_token_ids_start_idx + len(self.response_token_ids)
                
                    # Make pytorch loss function ignore all tokens up through the end of the response key
                    batch["labels"][i, response_token_ids_last_idx:response_token_ids_end_idx] = self.ignore_index
                    
                    if self.multiple_responses:
                        response_token_ids_last_idx = response_token_ids_end_idx + self.resposen_num_tokens
                
            if self.resposen_num_tokens and response_token_ids_end_idx:
                response_token_ids_last_idx = response_token_ids_end_idx + self.resposen_num_tokens
                
                if response_token_ids_last_idx < batch["labels"][i].shape[0]:
                    batch["labels"][i, response_token_ids_last_idx:] = self.ignore_index
                
                    
            if response_token_ids_start_idx is None:
                warnings.warn(
                    f"Could not find response key `{self.response_template}` in the following instance: "
                    f"{self.tokenizer.decode(batch['input_ids'][i])}. This instance will be ignored in loss "
                    "calculation. Note, if this happens often, consider increasing the `max_length`.",
                    UserWarning,
                )
                batch["labels"][i, :] = self.ignore_index
            
        return batch



class JigsawDataset(Dataset):
    def __init__(
        self,
        df,
        tokenizer,
        n_examples=4,
        choices_v=CHOICES_V,
        response_template=RESPONSE_TEMPLATE,
        rnd_examples_position=True,
        truncation_max_length=MAX_TOKEN_LEN,
    ):
        self.df = df
        self.df['body'] = self.df['body'].str.strip()
        
        self.tokenizer = tokenizer
        self.choices_v = choices_v
        self.n_examples = n_examples
        self.rnd_examples_position = rnd_examples_position
        self.response_template = response_template
        self.truncation_max_length = truncation_max_length
        
        self.positive_examples_d = df[['rule', 'body']][df['rule_violation'] == 1].groupby('rule').agg(list).to_dict()['body']
        self.negative_examples_d = df[['rule', 'body']][df['rule_violation'] == 0].groupby('rule').agg(list).to_dict()['body']
        
        return None
    
    
    def build_prompt(self, row, examples_v):
        rule_str = row['rule'].strip()
        body_str = row['body'].strip()
                    
        sys_prompt = f"You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes or No.\nRule: {rule_str}"

        # Format as a chat conversation using the model's template
        messages = [
            {"role": "system", "content": sys_prompt.strip()},
        ]

        example_idx = 0
        for example_str, rule_violation in examples_v:
            label = self.choices_v[ rule_violation ]
            example_idx += 1
            messages.extend(
                [
                    {"role": "user", "content": f'{example_idx}. ```{example_str}```'},
                    {"role": "assistant", "content": f"{self.response_template}{label}"},
                ]
            )

        messages.append(
            {"role": "user", "content": f'{example_idx+1}. ```{body_str}```'},
        )

        prompt = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + self.response_template
        
        if "rule_violation" in row:
            label = self.choices_v[ row["rule_violation"] ]
            prompt = prompt + label
            
        return prompt
    
    
    def get_rnd_examples(self, row):
        rule = row['rule']
        body = row['body']
        
        examples_v = []
        
        n_positive = random.randint(0, self.n_examples)
        
        while len(examples_v) < n_positive:
            for i_try in range(5):
                example = random.choice( self.positive_examples_d[rule] )
                element = (example, 1)
                if (example != body) and (element not in examples_v):
                    break
                    
            examples_v.append(element)
            
                
        while len(examples_v) < self.n_examples:
            for i_try in range(5):
                example = random.choice( self.negative_examples_d[rule] )
                element = (example, 0)
                if (example != body) and (element not in examples_v):
                    break
                    
            examples_v.append(element)
        
        if self.rnd_examples_position:
            random.shuffle( examples_v )
            
        return examples_v
    
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        examples_v = self.get_rnd_examples(row)
        
        truncation_ok = False
        for i in range(50):
            prompt = self.build_prompt(row, examples_v)
        
            inputs = self.tokenizer(
                prompt,
                add_special_tokens=False,
            )

            if len(inputs['input_ids']) <= self.truncation_max_length:
                truncation_ok = True
                break
                
            else:
                w_v = row['body'].split(' ')
                if (len(w_v) > 1) and len(w_v[-1]) < 30:
                    row['body'] = ' '.join(w_v[:-1])
                elif (len(w_v) > 1) and len(w_v[-1]) >= 30:
                    row['body'] = row['body'][:-15]
                    
                else:
                    break
                
                print(len(inputs['input_ids']), row['body'])
        if not truncation_ok:
            inputs = self.tokenizer(
                prompt,
                add_special_tokens=False,
                truncation=True,
                max_length=self.truncation_max_length
            )

        return inputs
    
        
    def __len__(self):
        return len(self.df)


def preprocess_logits_for_metrics(logits, labels):
    global YES_ID, NO_ID
    
    next_possible_logit = logits[:,:-1, [NO_ID, YES_ID]]
    f_v = (labels[:,1:] >= 0)
    
    next_preds_v = []
    for i, f in enumerate(f_v):
        next_preds = next_possible_logit[i][f]
        n_extra_logits = (N_EXAMPLES + 1) - next_preds.shape[0]
        
        if n_extra_logits > 0:
            print(f'WARNNG, preprocess_logits_for_metrics: missing preds. {next_preds.shape=} {N_EXAMPLES=} {n_extra_logits=}')
            next_preds = torch.concatenate(
                [
                    next_preds,
                    torch.ones(
                        (n_extra_logits, next_preds.shape[1]),
                        dtype=next_preds.dtype,
                        device=next_preds.device,
                    ),
                ],
                axis=0,
            )
        
        next_preds_v.append(next_preds)
        
    next_preds_v = torch.stack(next_preds_v).softmax(-1)
    return next_preds_v
    

def compute_metrics(pred_labels):
    global YES_ID, NO_ID
    
    pred, labels = pred_labels
    
    
    if len(labels.shape) == 2:
        selected_labels = []
        for sample_lbl_v in labels:
            sample_lbl_v = sample_lbl_v[sample_lbl_v >= 0]

            n_extra_labels = (N_EXAMPLES + 1) - sample_lbl_v.shape[0]
            if n_extra_labels > 0:
                print(f'WARNNG, compute_metrics: missing labels. {sample_lbl_v.shape=} {N_EXAMPLES=} {n_extra_labels=}')

                sample_lbl_v = np.concatenate(
                    [
                        sample_lbl_v,
                        np.random.choice(
                            [YES_ID, NO_ID],
                            size=n_extra_labels, 
                        ),
                    ],
                    axis=0,
                )

            selected_labels.append(sample_lbl_v)

        selected_labels = np.array(selected_labels)
        y_true = (selected_labels == YES_ID).astype(np.int64) # Yes labels
        
    else:
        y_true = labels.astype(np.int64)[:,None]
        pred = pred[:, None, :]
        
    n_preds = pred.shape[1] 
            
    y_scores = pred[..., 1] # Yes scores
    y_pred = (y_scores > 0.5).astype(np.int64)
    
    metrcs_d = {}
    for idx in range(n_preds):
        if idx == n_preds-1:
            sample_tag = ''
        else:
            sample_tag = f'_ex{idx}'
            
        metrcs_d[f"ACC{sample_tag}"] = accuracy_score(y_true[:,idx], y_pred[:,idx])
        metrcs_d[f"AUC{sample_tag}"] = roc_auc_score(y_true[:,idx], y_scores[:,idx])


    return metrcs_d



def load_model_and_tokenizer(device_id):
    trn_start_lora = TRN_START_LORA_V[device_id]
    model_name = MODEL_NAME_V[device_id]
    print(f'Loading basemodel: {model_name}')
    
    if UNSLOTH_TRAINING:
        base_model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=MAX_TOKEN_LEN,   # Context length - can be longer, but uses more memory
            load_in_4bit=True,     # 4bit uses much less memory
            load_in_8bit=False,    # A bit more accurate, uses 2x memory
            full_finetuning=False, # We have full finetuning now!
        )
        
        model = FastLanguageModel.get_peft_model(
            base_model,
            r = LORA_RANK,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ],
            lora_alpha=LORA_ALPHA,  # Best to choose alpha = rank or rank*2
            lora_dropout=0., # Supports any, but = 0 is optimized
            bias ="none",    # Supports any, but = "none" is optimized
            use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
            random_state=SEED + device_id,
            use_rslora=False,   # We support rank stabilized LoRA
            loftq_config=None,  # And LoftQ
        )
        
        if trn_start_lora:
            print(f'Loading LORA: {trn_start_lora}')
            model.load_adapter(trn_start_lora, 'default')
        
    else:
        # Initialize tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.padding_side = "left"  # Important for causal language models
        
        # Load the base model
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            trust_remote_code=True,
            #device_map="auto",  # Automatically distribute model across available GPUs
        )
        
        # Apply LoRA to the model
        if trn_start_lora:
            print(f'Loading LORA: {trn_start_lora}')
            
            model = peft.PeftModel.from_pretrained(
                base_model,
                trn_start_lora,
                is_trainable=True,
            )
        
        else:
            torch.manual_seed(SEED + device_id)

            # Configure LoRA parameters
            lora_config = LoraConfig(
                r=LORA_RANK,  # Rank of the update matrices
                lora_alpha=LORA_ALPHA,  # Alpha parameter for LoRA scaling
                lora_dropout=0.05,  # Dropout probability for LoRA layers
                task_type=TaskType.CAUSAL_LM,
                bias='none',  # Don't train bias terms
                # Target the attention and MLP modules of the transformer
                target_modules=[
                    "q_proj",
                    "k_proj",
                    "v_proj",
                    "o_proj",
                    "gate_proj",
                    "up_proj",
                    "down_proj",
                ]
            )

            model = get_peft_model(base_model, lora_config)

    model.print_trainable_parameters()
    
    return model, tokenizer


def train_on_device(device_id=0):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    
    OUTPUT_DIR = OUTPUT_DIR_V[device_id]
    MODEL_OUTPUT_PATH = MODEL_OUTPUT_PATH_V[device_id]
    
    if os.path.exists(MODEL_OUTPUT_PATH):
        print(f'⚠️ Removing: {MODEL_OUTPUT_PATH}')
        os.system(f'rm -r {MODEL_OUTPUT_PATH}')
    
    global NO_ID
    global YES_ID
    
    model, tokenizer = load_model_and_tokenizer(device_id)
    
    data_collator = DataCollatorForCompletionOnlyLM(
        RESPONSE_TEMPLATE,
        tokenizer=tokenizer,
        multiple_responses=TRAIN_ALL_ANSWERS,
    )

    df_trn, df_val, df_fit = load_trn_val_fit_df(ds_seed=DS_SEED + device_id)
    
    NO_ID = tokenizer.encode(CHOICES_V[0], add_special_tokens=False)[0]
    YES_ID = tokenizer.encode(CHOICES_V[1], add_special_tokens=False)[0]
    
    print(f'{NO_ID=} {YES_ID=}')
    print()
    print(f'{len(df_trn)=}')
    if df_val is None:
        print(f'{df_val=}')
        ds_val = None
    else:
        print(f'{len(df_val)=}')
        ds_val = JigsawDataset(df_val, tokenizer=tokenizer, n_examples=N_EXAMPLES, choices_v=CHOICES_V, rnd_examples_position=True, truncation_max_length=MAX_TOKEN_LEN)
        
    ds_trn = JigsawDataset(df_trn, tokenizer=tokenizer, n_examples=N_EXAMPLES, choices_v=CHOICES_V, rnd_examples_position=True, truncation_max_length=MAX_TOKEN_LEN)
    
    
    LOGGING_STRATEGY = ['epoch', 'steps'][1]
    if ds_val is None:
        SAVE_STRATEGY = 'no'
        EVAL_STRATEGY = 'no'
    else:
        SAVE_STRATEGY = LOGGING_STRATEGY
        EVAL_STRATEGY = LOGGING_STRATEGY


    LOGGING_STEPS = len(ds_trn) // (TRAIN_BS * GRAD_ACC_NUM * N_EVALS_PER_EPOCH * torch.cuda.device_count())
    SAVE_TOTAL_LIMIT = N_EVALS_PER_EPOCH + 1

    SAVE_STEPS = LOGGING_STEPS

    
    print()
    print('------ CFG----------')
    print(f'DEVICE_ID={device_id}')
    print(f'{EVAL_STRATEGY=}')
    print(f'{LOGGING_STEPS=}')
    print(f'{SAVE_STEPS=}')
    print(f'{SAVE_TOTAL_LIMIT=}')
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')
    print(f'{MAX_TOKEN_LEN=}')
    print(f'{len(ds_trn)=}')
    if ds_val is None:
        print(f'{ds_val=}')
    else:
        print(f'{len(ds_val)=}')
    print('----------------')
    print()
    
    # Set up training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        
        num_train_epochs=EPOCH,
        optim="paged_adamw_8bit",  # 8-bit optimizer for memory efficiency
        lr_scheduler_type=SCHEDULER_TYPE,
        warmup_ratio=0.1,  # Warm up learning rate over 10% of steps
        learning_rate=LR,
        weight_decay=0.01,
    
        # Use BF16 if available, otherwise FP16
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
    
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACC_NUM,
        gradient_checkpointing=True,  # Save memory with gradient checkpointing
        gradient_checkpointing_kwargs={"use_reentrant": False},
        group_by_length=False,
        seed=SEED + device_id,
        remove_unused_columns=False,  # Keep all columns in the dataset
        
        report_to="tensorboard",
        metric_for_best_model="AUC",
        greater_is_better=True,

        load_best_model_at_end=(ds_val is not None),
        save_total_limit=SAVE_TOTAL_LIMIT,

        logging_strategy=LOGGING_STRATEGY,
        eval_strategy=EVAL_STRATEGY,
        save_strategy=SAVE_STRATEGY,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
    )
    
    # Initialize trainer
    trainer = Trainer(
        model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=ds_trn,
        eval_dataset=ds_val,
        data_collator=data_collator,
        compute_metrics=compute_metrics if ds_val is not None else None,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics if ds_val is not None else None,
    )
    
    # Start training
    trainer_output = trainer.train()
    
    # Save the final model
    trainer.save_model(MODEL_OUTPUT_PATH)
    
if __name__ == '__main__':
    train_on_device()


## qwen_embeddings_inference.py

In [ ]:
%%writefile qwen_embeddings_inference.py

import os
import random
import pandas as pd
import numpy as np
import random
import multiprocessing as mp
import xgboost as xgb
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


from qwen_constants import *
from qwen_utils import load_trn_val_fit_df, save_obj


def fit_gbm_models(X_trn, y_trn, X_tst, n_splits=4, seed=2434, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)

    print(f'LGBM: {X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'LGBM: fitting fold: {i_fold}')

        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1
        }
        if seed:
            params['seed'] = seed + i_fold
            
        callbacks_v = [lgb.early_stopping(stopping_rounds=50)]

        if verbose:
            callbacks_v.append(lgb.log_evaluation(period=50))
            
        # Entrenar
        model = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=callbacks_v
        )

        # Predicciones
        y_fold_pred = model.predict(X_tst, num_iteration=model.best_iteration)
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)

    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v

def fit_xgb_models(X_trn, y_trn, X_tst, n_splits=4, seed=234, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=4342)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        
    print(f'XGB: {X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'XGB: fitting fold: {i_fold}')
        
        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]
        
        model = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.1,
            max_depth=10,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            early_stopping_rounds=25, 
            metric='auc',
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=verbose
        )
        # Predicciones
        y_fold_pred = model.predict(X_tst, iteration_range=(0, model.best_iteration + 1))
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)
        
    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v

def fit_and_predict_embeddings(fit_samples_df, tst_samples_df):
    all_tst_preds_v = np.zeros( (len(tst_samples_df), 2) )
    n_clf = 0
    for i_rule, rule in enumerate(tst_samples_df.rule.unique()):
        f_fit = (fit_samples_df['rule'] == rule)
        f_pred = (tst_samples_df.rule.values == rule)

        if 'RB_LG' in OUTPUT_PREDICTOR_CFG:
            print(f'Fitting: LogisticRegression DS[rule={i_rule}]')
            clf = LogisticRegression(max_iter=1000, C=1)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1

        if 'RB_SVC' in OUTPUT_PREDICTOR_CFG:
            print(f'Fitting: SVC rule: DS[rule={i_rule}]')
            clf = SVC(kernel="linear", C=2, probability=True, random_state=42)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1


        if 'RB_LGBM' in OUTPUT_PREDICTOR_CFG:
            print(f'🟠 Fitting: LGBM rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_gbm_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1

        if 'RB_XGB' in OUTPUT_PREDICTOR_CFG:
            print(f'🟢 Fitting: XGB rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_xgb_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1
            
    if 'FDS_LG' in OUTPUT_PREDICTOR_CFG:
        print('Fitting: LogisticRegression FullDS')
        clf = LogisticRegression(max_iter=1000, C=2)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1

    if 'FDS_SVC' in OUTPUT_PREDICTOR_CFG:
        print('Fitting: SVC FullDS')
        clf =SVC(kernel="linear", C=2, probability=True, random_state=42)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1


    if 'FDS_LGBM' in OUTPUT_PREDICTOR_CFG:
        print('🟠 Fitting: LGBM FullDS')
        all_tst_preds_v += fit_gbm_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=8
        )[0]
        n_clf += 1
        
        
    if 'FDS_XGB' in OUTPUT_PREDICTOR_CFG:
        print('🟢 Fitting: XGB FullDS')
        all_tst_preds_v += fit_xgb_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=4
        )[0]
        n_clf += 1

    all_tst_preds_v = all_tst_preds_v / n_clf

    return all_tst_preds_v



def create_prompts(df, tokenizer):
    prompts_v = []
    for i, row in df.iterrows():
        # Define system prompt for the classification task

        sys_prompt = f"You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes or No.\nRule: {row.rule.strip()}"
        
        # Format as a chat conversation using the model's template
        messages = [
            {"role": "system", "content": sys_prompt.strip()},
            {"role": "user", "content": '1. ```{}```'.format(row.body.strip())},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Violation:"

        prompts_v.append(prompt)
    return prompts_v


def generate_embeddings(llm, df, lora_path):
    from vllm.lora.request import LoRARequest
    
    all_outputs = llm.embed(
        df['prompts'].values,
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, lora_path)
    )

    all_embeddings_v = []
    for output in all_outputs:
        all_embeddings_v.append(
            np.array(output.outputs.embedding)
        )
        
    df['embeddings'] = all_embeddings_v
    
    return all_embeddings_v


def create_clf_dataset(df):
    dfs_to_concat_v = []
    if ('positive_example_1' in df.columns) and ('positive_example_2' in df.columns):
        all_positives_df = pd.concat(
            [
                df[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'}),
                df[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_positives_df['rule_violation'] = 1
        dfs_to_concat_v.append(all_positives_df)
    
    
    if ('negative_example_1' in df.columns) and ('negative_example_2' in df.columns):
        all_negatives_df = pd.concat(
            [
                df[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'}),
                df[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_negatives_df['rule_violation'] = 0
        dfs_to_concat_v.append(all_negatives_df)
    
    
    if ('body' in df.columns) and ('rule_violation' in df.columns):
        all_bodies_df = df[['rule', 'body', 'rule_violation']].copy()
        dfs_to_concat_v.append(all_bodies_df)
        
        
    if len(dfs_to_concat_v) > 1:
        all_examples_df = pd.concat(
            dfs_to_concat_v,
            axis=0
        ).drop_duplicates()
        
    elif len(dfs_to_concat_v) == 1:
        all_examples_df = dfs_to_concat_v[0].drop_duplicates()
        
    else:
        raise Exception('No data found')
        
    return all_examples_df.reset_index(drop=True)


def run_inference_on_device(fit_samples_df, tst_samples_df, base_model_path, lora_path):
    import vllm

    llm = vllm.LLM(
        base_model_path,
        quantization='awq' if 'awq' in base_model_path else ('gptq' if 'gptq' in base_model_path else None),
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2836,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        task='embed',
        max_lora_rank=max(16, LORA_RANK),
    )
        
    tokenizer = llm.get_tokenizer()

    fit_samples_df['prompts'] = create_prompts(fit_samples_df, tokenizer)
    fit_samples_embeddigs_v = generate_embeddings(llm, fit_samples_df, lora_path)

    tst_samples_df['prompts'] = create_prompts(tst_samples_df, tokenizer)
    tst_samples_embeddigs_v = generate_embeddings(llm, tst_samples_df, lora_path)
    return fit_samples_df, tst_samples_df


def worker(device_id, fit_samples_df, tst_samples_df, base_model_path, lora_path, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    os.environ["VLLM_USE_V1"] = "0"
    
    print(f"[Worker {device_id}] 🔥 Running on GPU {device_id}, {len(fit_samples_df)=} {len(tst_samples_df)=} {lora_path=}")
    fit_samples_df, tst_samples_df = run_inference_on_device(fit_samples_df, tst_samples_df, base_model_path, lora_path)
    return_dict[f'fit_samples_df_device{device_id}'] = fit_samples_df
    return_dict[f'tst_samples_df_device{device_id}'] = tst_samples_df
    print(f"[Worker {device_id}] ✅ Inference OK!")

def many_models_inference():
    n_devices = 2
    manager = mp.Manager()
    return_dict = manager.dict()
    
    df_tst = pd.read_csv(INFERENCE_DATASET)
    
    p_v = []
    for device_id in range(n_devices):
        device_fit_samples_df = load_trn_val_fit_df(DS_SEED + device_id)[-1] # Fitting with val samples
        device_tst_samples_df = df_tst[['row_id', 'rule', 'subreddit', 'body']].copy()
        
        p = mp.Process(
            target=worker,
            args=(
                device_id,
                device_fit_samples_df,
                device_tst_samples_df,
                MODEL_NAME_INF_V[device_id],
                MODEL_OUTPUT_PATH_V[device_id],
                return_dict,
            )
        )
        p_v.append(p)
        p.start()
        
    for p in p_v:
        p.join()
    
    print('Embeddings inference Finished!!!')
    
    device_fit_samples_df_v = [return_dict[f'fit_samples_df_device{device_id}'] for device_id in range(n_devices)]
    device_tst_samples_df_v = [return_dict[f'tst_samples_df_device{device_id}'] for device_id in range(n_devices)]
    
    #save_obj(device_fit_samples_df_v, 'device_fit_samples_df_v.pickle')
    #save_obj(device_tst_samples_df_v, 'device_tst_samples_df_v.pickle')

    for device_id, (fit_samples_df, tst_samples_df) in enumerate(zip(device_fit_samples_df_v, device_tst_samples_df_v)):
        print(f'☢️ Fitting embeddings CLFs. For {device_id=}')
        all_tst_preds_v = fit_and_predict_embeddings(fit_samples_df, tst_samples_df)
        tst_samples_df["pred"] = all_tst_preds_v[:, 1]

        df = tst_samples_df.copy()
        df['rule_violation'] = df["pred"]
        df[['row_id', 'rule_violation']].to_csv(OUTPUT_CSV_V[device_id], index=False)
        print(f"✅ Saved: {OUTPUT_CSV_V[device_id]}")
        
        if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
            from sklearn.metrics import accuracy_score, roc_auc_score
            df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
            AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=df.rule_violation.values)
            ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=df.rule_violation.values > 0.5)

            print('Test GT Found, computing Final score:')
            print(f'-Test {AUC=:0.4f}')
            print(f'-Test {ACC=:0.4f}')

    
    if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
        print('ENSEMBLE:')
        ens_df = df.copy()
        ens_df['rule_violation'] = np.mean([df["pred"].values for df in device_tst_samples_df_v], axis=0)
        
        from sklearn.metrics import accuracy_score, roc_auc_score
        df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
        AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=ens_df.rule_violation.values)
        ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=ens_df.rule_violation.values > 0.5)

        print('Test GT Found, computing Final score:')
        print(f'-Test {AUC=:0.4f}')
        print(f'-Test {ACC=:0.4f}')
        
    return None

def single_model_inference(lora_model_path, output_csv='./submission.csv'):
    print('Inference settings:')
    print(f'{MODEL_NAME_INF_V=}')
    print(f'{lora_model_path=}')
    print(f'{output_csv=}')
    print()
    
    manager = mp.Manager()
    return_dict = manager.dict()
    
    fit_samples_df = load_trn_val_fit_df(DS_SEED)[-1] # Fitting with val samples
    
    df_tst = pd.read_csv(INFERENCE_DATASET)
    tst_samples_df = df_tst[['row_id', 'rule', 'subreddit', 'body']].copy()

    fit_mid = len(fit_samples_df) // 2
    tst_mid = len(tst_samples_df) // 2
    
    p0 = mp.Process(
        target=worker,
        args=(
            0,
            fit_samples_df.iloc[:fit_mid].reset_index(drop=True),
            tst_samples_df.iloc[:tst_mid].reset_index(drop=True),
            MODEL_NAME_INF_V[0],
            lora_model_path,
            return_dict,
        )
    )
    p1 = mp.Process(
        target=worker,
        args=(
            1,
            fit_samples_df.iloc[fit_mid:].reset_index(drop=True),
            tst_samples_df.iloc[tst_mid:].reset_index(drop=True),
            MODEL_NAME_INF_V[1],
            lora_model_path,
            return_dict,
        )
    )
    p0.start()
    p1.start()
    p0.join()
    p1.join()
    
    fit_samples_df = pd.concat(
        [
            return_dict['fit_samples_df_device0'],
            return_dict['fit_samples_df_device1']
        ],
        ignore_index=True
    ).reset_index(drop=True)
    
    tst_samples_df = pd.concat(
        [
            return_dict['tst_samples_df_device0'],
            return_dict['tst_samples_df_device1']
        ],
        ignore_index=True
    ).reset_index(drop=True)
    
    #save_obj(fit_samples_df, 'fit_samples_df.pickle')
    #save_obj(tst_samples_df, 'tst_samples_df.pickle')

    print('☢️ Fitting embeddings CLFs.')
    all_tst_preds_v = fit_and_predict_embeddings(fit_samples_df, tst_samples_df)
    tst_samples_df["pred"] = all_tst_preds_v[:, 1]

    df = tst_samples_df.copy()
    df['rule_violation'] = df["pred"]
    df[['row_id', 'rule_violation']].to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")
    
    if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
        from sklearn.metrics import accuracy_score, roc_auc_score
        df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
        AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=df.rule_violation.values)
        ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=df.rule_violation.values > 0.5)

        print('Test GT Found, computing Final score:')
        print(f'-Test {AUC=:0.4f}')
        print(f'-Test {ACC=:0.4f}')
    return None


def models_merge(loras_to_merge_v, output_dir):
    import shutil
    from glob import glob
    import torch
    import safetensors.torch
    from collections import OrderedDict
    
    print('💈 Creating merge of LORAs ...')
    
    if os.path.exists(output_dir):
        os.system(f'rm -r {output_dir}')
    
    os.makedirs(output_dir, exist_ok=True)

    # Copy model structure
    for file_path in glob(os.path.join(loras_to_merge_v[0], '*')):
        if os.path.isdir(file_path) or file_path.endswith('adapter_model.safetensors'):
            continue
        shutil.copy(file_path, output_dir)
        
    merged_ckpt_d = OrderedDict()
    for lora_filepath in loras_to_merge_v:
        print(f'Reading: {lora_filepath}')
        ckpt_data_d = safetensors.torch.load_file(
            os.path.join(lora_filepath, 'adapter_model.safetensors')
        )
        
        for k, v in ckpt_data_d.items():
            if k not in merged_ckpt_d.keys():
                merged_ckpt_d[k] = [v]
                
            else:
                merged_ckpt_d[k].append(v)
                
    for k in merged_ckpt_d.keys():
        merged_ckpt_d[k] = torch.stack(
            merged_ckpt_d[k]
        ).mean(0)
        
    safetensors.torch.save_file(
        merged_ckpt_d,
        os.path.join(output_dir, 'adapter_model.safetensors')
    )
    print(f'💈 Merge of LORAs completed: {output_dir}')
    return None


if __name__ == "__main__":
    if LORAS_MERGE:
        merge_proc = mp.Process(
            target=models_merge,
            args=(
                MODEL_OUTPUT_PATH_V,
                MERGE_OUTPUT_DIR,
            )
        )
        merge_proc.start()
        merge_proc.join()
        
        single_model_inference(MERGE_OUTPUT_DIR, output_csv=MERGE_OUTPUT_CSV)
        
    else:
        many_models_inference()

## qwen_fit_all.py

In [ ]:
%%writefile qwen_fit_all.py

import multiprocessing as mp

def worker(device_id):
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    
    from qwen_train import train_on_device
    train_on_device(device_id)


if __name__ == '__main__':
    manager = mp.Manager()
    
    p0 = mp.Process(
        target=worker,
        args=(
            0,
        )
    )
    p1 = mp.Process(
        target=worker,
        args=(
            1,
        )
    )
    p0.start()
    p1.start()
    p0.join()
    p1.join()

## qwen_constants.py: Qwen2.5 14b x2

In [ ]:
%%writefile qwen_constants.py

import os

MODEL_CFG = ['7b-8b', '7b', '8b', '14b'][-1]
LORAS_MERGE = False

MODEL_NAME_V = None
MODEL_NAME_INF_V = None

if MODEL_CFG == '14b':
    MODEL_NAME = "../input/m/shelterw/qwen2.5/transformers/qwen2.5-14b-instruct-unsloth-bnb-4bit/1"
    MODEL_NAME_INF = "../input/qwen2.5/transformers/14b-instruct-gptq-int4/1"

    if LORAS_MERGE:
        0/0
        
    else:
        TRN_START_LORA_V = [
            "/kaggle/input/jigsaw-t49-c469-qwen2-5-14b-instruct-us4bit",
            "/kaggle/input/jigsaw-t50-c313-qwen2-5-14b-instruct-us4bit",
        ]
        
        OUTPUT_CSV_V = [
            './submission_ttt_WQ25_14b_0.csv',
            './submission_ttt_WQ25_14b_1.csv',
        ]
        
    OUTPUT_DIR_V = [
        "./ttt_unsloth_output_WQ25_14b_0",
        "./ttt_unsloth_output_WQ25_14b_1",
    ]
    
    EPOCH = 1  # Training epochs
    LR = 2e-4  # Learning rate
    SCHEDULER_TYPE = ['cosine', 'linear'][0]
    DS_SEED = 434
    SEED = 3433  # Random seed for reproducibility
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')

if MODEL_NAME_V is None:
    MODEL_NAME_V = [MODEL_NAME, MODEL_NAME]
    del MODEL_NAME
    
if MODEL_NAME_INF_V is None:
    MODEL_NAME_INF_V = [MODEL_NAME_INF, MODEL_NAME_INF]
    del MODEL_NAME_INF
    
MODEL_OUTPUT_PATH_V = [os.path.join(out_dir, 'best_model') for out_dir in OUTPUT_DIR_V]
INFERENCE_DATASET = "../input/jigsaw-agile-community-rules/test.csv"

UNSLOTH_TRAINING = True

if MODEL_CFG in ['7b', '8b', '7b-8b']:
    # small model
    LORA_RANK = 32
    LORA_ALPHA = 32
    TRAIN_BS = 8 * 2
    EVAL_BS = 8 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768
    
    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00 #0.40
    
elif MODEL_CFG == '14b':
    LORA_RANK = 16
    LORA_ALPHA = 16
    TRAIN_BS = 3 * 2
    EVAL_BS = 3 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768

    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')


IS_DEBUG = False  # Debug mode with small dataset
N_FOLDS = 5  # Number of cross-validation folds
FOLD = None  # Current fold to train
N_EXAMPLES = 0


TRAIN_ALL_ANSWERS = True

CHOICES_V = [" No", " Yes"]
RESPONSE_TEMPLATE = "Violation:"

OUTPUT_PREDICTOR_CFG = [
    #'RB_LG',
    'RB_SVC',
    'RB_LGBM',
    #'RB_XGB',
    
    #'FDS_LG',
    #'FDS_SVC',
    #'FDS_LGBM',
    #'FDS_XGB',
]

for out_dir in OUTPUT_DIR_V:
    os.makedirs(out_dir, exist_ok=True)

for init_lora in TRN_START_LORA_V:
    assert (init_lora is None) or os.path.exists(init_lora), init_lora

for model_name in MODEL_NAME_V:
    assert (model_name is None) or os.path.exists(model_name), model_name
    
for model_name_inf in MODEL_NAME_INF_V:
    assert (model_name_inf is None) or os.path.exists(model_name_inf), model_name_inf

In [ ]:
%%time
!python qwen_fit_all.py

In [ ]:
%%time
!python qwen_embeddings_inference.py

In [ ]:
!head submission_ttt_WQ25_14b_0.csv

In [ ]:
!head submission_ttt_WQ25_14b_1.csv

## qwen_constants.py: Qwen3 14b x2

In [ ]:
%%writefile qwen_constants.py

import os

MODEL_CFG = ['7b-8b', '7b', '8b', '14b'][-1]
LORAS_MERGE = False

MODEL_NAME_V = None
MODEL_NAME_INF_V = None

if MODEL_CFG == '14b':
    MODEL_NAME = "../input/qwen3-14b-unsloth-bnb-4bit"
    MODEL_NAME_INF = "../input/qwen-3/transformers/14b-awq/1"

    if LORAS_MERGE:
        0/0
        
    else:
        TRN_START_LORA_V = [
            "/kaggle/input/jigsaw-t73-c469-qwen3-14b-us4bit",
            "/kaggle/input/jigsaw-t74-c469-qwen3-14b-us4bit",
        ]
        
        OUTPUT_CSV_V = [
            './submission_ttt_WQ3_14b_0.csv',
            './submission_ttt_WQ3_14b_1.csv',
        ]
        
    OUTPUT_DIR_V = [
        "./ttt_unsloth_output_WQ3_14b_0",
        "./ttt_unsloth_output_WQ3_14b_1",
    ]
    
    EPOCH = 1  # Training epochs
    LR = 2e-4  # Learning rate
    SCHEDULER_TYPE = ['cosine', 'linear'][0]
    DS_SEED = 3937
    SEED = 12323  # Random seed for reproducibility
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')

if MODEL_NAME_V is None:
    MODEL_NAME_V = [MODEL_NAME, MODEL_NAME]
    del MODEL_NAME
    
if MODEL_NAME_INF_V is None:
    MODEL_NAME_INF_V = [MODEL_NAME_INF, MODEL_NAME_INF]
    del MODEL_NAME_INF
    
MODEL_OUTPUT_PATH_V = [os.path.join(out_dir, 'best_model') for out_dir in OUTPUT_DIR_V]
INFERENCE_DATASET = "../input/jigsaw-agile-community-rules/test.csv"

UNSLOTH_TRAINING = True

if MODEL_CFG in ['7b', '8b', '7b-8b']:
    # small model
    LORA_RANK = 32
    LORA_ALPHA = 32
    TRAIN_BS = 8 * 2
    EVAL_BS = 8 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768
    
    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00 #0.40
    
elif MODEL_CFG == '14b':
    LORA_RANK = 16
    LORA_ALPHA = 16
    TRAIN_BS = 3 * 2
    EVAL_BS = 3 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768

    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')


IS_DEBUG = False  # Debug mode with small dataset
N_FOLDS = 5  # Number of cross-validation folds
FOLD = None  # Current fold to train
N_EXAMPLES = 0


TRAIN_ALL_ANSWERS = True

CHOICES_V = [" No", " Yes"]
RESPONSE_TEMPLATE = "Violation:"

OUTPUT_PREDICTOR_CFG = [
    #'RB_LG',
    'RB_SVC',
    'RB_LGBM',
    #'RB_XGB',
    
    #'FDS_LG',
    #'FDS_SVC',
    #'FDS_LGBM',
    #'FDS_XGB',
]

for out_dir in OUTPUT_DIR_V:
    os.makedirs(out_dir, exist_ok=True)

for init_lora in TRN_START_LORA_V:
    assert (init_lora is None) or os.path.exists(init_lora), init_lora

for model_name in MODEL_NAME_V:
    assert (model_name is None) or os.path.exists(model_name), model_name
    
for model_name_inf in MODEL_NAME_INF_V:
    assert (model_name_inf is None) or os.path.exists(model_name_inf), model_name_inf

In [ ]:
%%time
!python qwen_fit_all.py

In [ ]:
%%time
!python qwen_embeddings_inference.py

In [ ]:
!head submission_ttt_WQ3_14b_0.csv

In [ ]:
!head submission_ttt_WQ3_14b_1.csv

## qwen_constants.py: Qwen2.5 7b x2 (Merge)

In [ ]:
%%writefile qwen_constants.py

import os

MODEL_CFG = ['7b-8b', '7b', '8b', '14b'][1]
LORAS_MERGE = True

MODEL_NAME_V = None
MODEL_NAME_INF_V = None

if MODEL_CFG in '7b':
    MODEL_NAME = "../input/qwen2-5-7b-instruct-unsloth-bnb-4bit"
    MODEL_NAME_INF = "../input/qwen2.5/transformers/7b-instruct-gptq-int4/1"

    if LORAS_MERGE:
        TRN_START_LORA_V = [
            "/kaggle/input/jigsaw-t52-c235-qwen2-5-7b-instruct-us4bit",
            "/kaggle/input/jigsaw-t53-c235-qwen2-5-7b-instruct-us4bit",
        ]
        MERGE_OUTPUT_CSV = './submission_ttt_WQ25_7b_merge.csv'
        MERGE_OUTPUT_DIR = "./ttt_unsloth_WQ25_7b_merge"
        
    else:
        0/0
    
    OUTPUT_DIR_V = [
        "./ttt_unsloth_output_WQ25_7b_0",
        "./ttt_unsloth_output_WQ25_7b_1",
    ]
    
    EPOCH = 1  # Training epochs
    LR = 2e-4  # Learning rate
    SCHEDULER_TYPE = ['cosine', 'linear'][1]
    DS_SEED = 48692
    SEED = 9947  # Random seed for reproducibility


elif MODEL_CFG in '8b':
    0/0
    
elif MODEL_CFG == '14b':
    0/0
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')

if MODEL_NAME_V is None:
    MODEL_NAME_V = [MODEL_NAME, MODEL_NAME]
    del MODEL_NAME
    
if MODEL_NAME_INF_V is None:
    MODEL_NAME_INF_V = [MODEL_NAME_INF, MODEL_NAME_INF]
    del MODEL_NAME_INF
    
MODEL_OUTPUT_PATH_V = [os.path.join(out_dir, 'best_model') for out_dir in OUTPUT_DIR_V]
INFERENCE_DATASET = "../input/jigsaw-agile-community-rules/test.csv"

UNSLOTH_TRAINING = True

if MODEL_CFG in ['7b', '8b', '7b-8b']:
    # small model
    LORA_RANK = 32
    LORA_ALPHA = 32
    TRAIN_BS = 8 * 2
    EVAL_BS = 8 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768
    
    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00 #0.40
    
elif MODEL_CFG == '14b':
    LORA_RANK = 16
    LORA_ALPHA = 16
    TRAIN_BS = 3 * 2
    EVAL_BS = 3 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768

    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')


IS_DEBUG = False  # Debug mode with small dataset
N_FOLDS = 5  # Number of cross-validation folds
FOLD = None  # Current fold to train
N_EXAMPLES = 0


TRAIN_ALL_ANSWERS = True

CHOICES_V = [" No", " Yes"]
RESPONSE_TEMPLATE = "Violation:"

OUTPUT_PREDICTOR_CFG = [
    #'RB_LG',
    'RB_SVC',
    'RB_LGBM',
    #'RB_XGB',
    
    #'FDS_LG',
    #'FDS_SVC',
    #'FDS_LGBM',
    #'FDS_XGB',
]

for out_dir in OUTPUT_DIR_V:
    os.makedirs(out_dir, exist_ok=True)

for init_lora in TRN_START_LORA_V:
    assert (init_lora is None) or os.path.exists(init_lora), init_lora

for model_name in MODEL_NAME_V:
    assert (model_name is None) or os.path.exists(model_name), model_name
    
for model_name_inf in MODEL_NAME_INF_V:
    assert (model_name_inf is None) or os.path.exists(model_name_inf), model_name_inf

In [ ]:
%%time
!python qwen_fit_all.py

In [ ]:
%%time
!python qwen_embeddings_inference.py

In [ ]:
!head submission_ttt_WQ25_7b_merge.csv

## qwen_constants.py: Qwen3 8b x2 (Merge)

In [ ]:
%%writefile qwen_constants.py

import os

MODEL_CFG = ['7b-8b', '7b', '8b', '14b'][2]
LORAS_MERGE = True

MODEL_NAME_V = None
MODEL_NAME_INF_V = None

if MODEL_CFG in '7b':
    0/0

elif MODEL_CFG in '8b':
    MODEL_NAME = "../input/qwen3-8b-unsloth-bnb-4bit"
    MODEL_NAME_INF = "../input/qwen-3/transformers/8b-awq/1"

    if LORAS_MERGE:
        TRN_START_LORA_V = [
            "/kaggle/input/jigsaw-t59-c235-qwen3-8b-us4bit",
            "/kaggle/input/jigsaw-t60-c235-qwen3-8b-us4bit",
        ]
        MERGE_OUTPUT_CSV = './submission_ttt_WQ3_8b_merge.csv'
        MERGE_OUTPUT_DIR = "./ttt_unsloth_WQ3_8b_merge"
        
    else:
        0/0
    
    OUTPUT_DIR_V = [
        "./ttt_unsloth_output_WQ3_8b_0",
        "./ttt_unsloth_output_WQ3_8b_1",
    ]
    
    EPOCH = 1  # Training epochs
    LR = 2e-4  # Learning rate
    SCHEDULER_TYPE = ['cosine', 'linear'][1]
    DS_SEED = 70483
    SEED = 82836  # Random seed for reproducibility
    
elif MODEL_CFG == '14b':
    0/0
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')

if MODEL_NAME_V is None:
    MODEL_NAME_V = [MODEL_NAME, MODEL_NAME]
    del MODEL_NAME
    
if MODEL_NAME_INF_V is None:
    MODEL_NAME_INF_V = [MODEL_NAME_INF, MODEL_NAME_INF]
    del MODEL_NAME_INF
    
MODEL_OUTPUT_PATH_V = [os.path.join(out_dir, 'best_model') for out_dir in OUTPUT_DIR_V]
INFERENCE_DATASET = "../input/jigsaw-agile-community-rules/test.csv"

UNSLOTH_TRAINING = True

if MODEL_CFG in ['7b', '8b', '7b-8b']:
    # small model
    LORA_RANK = 32
    LORA_ALPHA = 32
    TRAIN_BS = 8 * 2
    EVAL_BS = 8 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768
    
    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00 #0.40
    
elif MODEL_CFG == '14b':
    LORA_RANK = 16
    LORA_ALPHA = 16
    TRAIN_BS = 3 * 2
    EVAL_BS = 3 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768

    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')


IS_DEBUG = False  # Debug mode with small dataset
N_FOLDS = 5  # Number of cross-validation folds
FOLD = None  # Current fold to train
N_EXAMPLES = 0


TRAIN_ALL_ANSWERS = True

CHOICES_V = [" No", " Yes"]
RESPONSE_TEMPLATE = "Violation:"

OUTPUT_PREDICTOR_CFG = [
    #'RB_LG',
    'RB_SVC',
    'RB_LGBM',
    #'RB_XGB',
    
    #'FDS_LG',
    #'FDS_SVC',
    #'FDS_LGBM',
    #'FDS_XGB',
]

for out_dir in OUTPUT_DIR_V:
    os.makedirs(out_dir, exist_ok=True)

for init_lora in TRN_START_LORA_V:
    assert (init_lora is None) or os.path.exists(init_lora), init_lora

for model_name in MODEL_NAME_V:
    assert (model_name is None) or os.path.exists(model_name), model_name
    
for model_name_inf in MODEL_NAME_INF_V:
    assert (model_name_inf is None) or os.path.exists(model_name_inf), model_name_inf

In [ ]:
%%time
!python qwen_fit_all.py

In [ ]:
%%time
!python qwen_embeddings_inference.py

In [ ]:
!head submission_ttt_WQ3_8b_merge.csv

## qwen_constants.py: Qwen3 7b x1. Qwen3 8b x1 (Merge)

In [ ]:
%%writefile qwen_constants.py

import os

MODEL_CFG = ['7b-8b', '7b', '8b', '14b'][0]
LORAS_MERGE = False

MODEL_NAME_V = None
MODEL_NAME_INF_V = None

if MODEL_CFG == '7b-8b':
    MODEL_NAME_V = [
        "../input/qwen2-5-7b-instruct-unsloth-bnb-4bit",
        "../input/qwen3-8b-unsloth-bnb-4bit",
    ]
    MODEL_NAME_INF_V = [
        "../input/qwen2.5/transformers/7b-instruct-gptq-int4/1",
        "../input/qwen-3/transformers/8b-awq/1",
    ]

    if LORAS_MERGE:
        0/0
        
    else:
        TRN_START_LORA_V = [
            "/kaggle/input/jigsaw-t52-c235-qwen2-5-7b-instruct-us4bit",
            "/kaggle/input/jigsaw-t60-c235-qwen3-8b-us4bit",
        ]
        
        OUTPUT_CSV_V = [
            './submission_ttt_7b_QW2.csv',
            './submission_ttt_8b_QW3.csv',
        ]
    
    OUTPUT_DIR_V = [
        "./ttt_unsloth_output_7b",
        "./ttt_unsloth_output_8b",
    ]
    
    EPOCH = 1  # Training epochs
    LR = 3e-4  # Learning rate
    SCHEDULER_TYPE = ['cosine', 'linear'][0]
    DS_SEED = 303987
    SEED = 73346  # Random seed for reproducibility
    

elif MODEL_CFG == '7b':
    0/0

elif MODEL_CFG == '8b':
    0/0
    
elif MODEL_CFG == '14b':
    0/0
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')

if MODEL_NAME_V is None:
    MODEL_NAME_V = [MODEL_NAME, MODEL_NAME]
    del MODEL_NAME
    
if MODEL_NAME_INF_V is None:
    MODEL_NAME_INF_V = [MODEL_NAME_INF, MODEL_NAME_INF]
    del MODEL_NAME_INF
    
MODEL_OUTPUT_PATH_V = [os.path.join(out_dir, 'best_model') for out_dir in OUTPUT_DIR_V]
INFERENCE_DATASET = "../input/jigsaw-agile-community-rules/test.csv"

UNSLOTH_TRAINING = True

if MODEL_CFG in ['7b', '8b', '7b-8b']:
    # small model
    LORA_RANK = 32
    LORA_ALPHA = 32
    TRAIN_BS = 8 * 2
    EVAL_BS = 8 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768
    
    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00 #0.40
    
elif MODEL_CFG == '14b':
    LORA_RANK = 16
    LORA_ALPHA = 16
    TRAIN_BS = 3 * 2
    EVAL_BS = 3 * 2
    GRAD_ACC_NUM = 1
    MAX_TOKEN_LEN = 768

    N_EVALS_PER_EPOCH = 2
    PUBLIC_TRN_DS_PROP = 0.01 #0.05
    TRAIN_DS_PROP = 0.93
    VAL_ON_TRAIN_DATA = True # False
    FIT_ON_VAL = False # True
    VAL_DS_SIZE = 0.00
    
else:
    raise NotImplementedError(f'{MODEL_CFG=} ???')


IS_DEBUG = False  # Debug mode with small dataset
N_FOLDS = 5  # Number of cross-validation folds
FOLD = None  # Current fold to train
N_EXAMPLES = 0


TRAIN_ALL_ANSWERS = True

CHOICES_V = [" No", " Yes"]
RESPONSE_TEMPLATE = "Violation:"

OUTPUT_PREDICTOR_CFG = [
    #'RB_LG',
    'RB_SVC',
    'RB_LGBM',
    #'RB_XGB',
    
    #'FDS_LG',
    #'FDS_SVC',
    #'FDS_LGBM',
    #'FDS_XGB',
]

for out_dir in OUTPUT_DIR_V:
    os.makedirs(out_dir, exist_ok=True)

for init_lora in TRN_START_LORA_V:
    assert (init_lora is None) or os.path.exists(init_lora), init_lora

for model_name in MODEL_NAME_V:
    assert (model_name is None) or os.path.exists(model_name), model_name
    
for model_name_inf in MODEL_NAME_INF_V:
    assert (model_name_inf is None) or os.path.exists(model_name_inf), model_name_inf

In [ ]:
#%%time
#!python qwen_fit_all.py

In [ ]:
#%%time
#!python qwen_embeddings_inference.py

In [ ]:
#!head submission_ttt_7b_QW2.csv

In [ ]:
#!head submission_ttt_8b_QW3.csv

# TTT-Triplet+Embedings

## settings_margin.py

In [ ]:
%%writefile settings_margin.py

import os
import re
import pickle
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
from urllib.parse import urlparse

from constants_margin import * 


def save_obj(obj, filename, verbose=True):
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)

    if verbose:
        print(f'Saved: {filename}')

    return None

def load_obj(filename, verbose=True):
    with open(filename, 'rb') as f:
        obj = pickle.load(f)

    if verbose:
        print(f'Loaded: {filename}')

    return obj


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))


def flip_rule(rule):
    str_replacement_v = [
        ("don[' ]t", ''),
        ('do not', ''),
        
        ('are not', ''),
        ("aren[ ']t", ''),
        
        ('is not', ''),
        ("isn[ ']t", ''),
        
        ('no', ''),
        (' +', ' '),
    ]
    
    for f, r in str_replacement_v:
        rule = re.sub(f, r, rule.lower())
        
    return rule.strip()
    

def create_clf_dataset(df):
    dfs_to_concat_v = []
    if ('positive_example_1' in df.columns) and ('positive_example_2' in df.columns):
        all_positives_df = pd.concat(
            [
                df[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'}),
                df[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_positives_df['rule_violation'] = 1
        dfs_to_concat_v.append(all_positives_df)
    
    
    if ('negative_example_1' in df.columns) and ('negative_example_2' in df.columns):
        all_negatives_df = pd.concat(
            [
                df[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'}),
                df[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_negatives_df['rule_violation'] = 0
        dfs_to_concat_v.append(all_negatives_df)
    
    
    if ('body' in df.columns) and ('rule_violation' in df.columns):
        all_bodies_df = df[['rule', 'body', 'rule_violation']].copy()
        dfs_to_concat_v.append(all_bodies_df)
        
        
    if len(dfs_to_concat_v) > 1:
        all_examples_df = pd.concat(
            dfs_to_concat_v,
            axis=0
        ).drop_duplicates()
        
    elif len(dfs_to_concat_v) == 1:
        all_examples_df = dfs_to_concat_v[0].drop_duplicates()
        
    else:
        raise Exception('No data found')
        
    return all_examples_df.reset_index(drop=True)


class JigsawDatasetTriplet():
    def __init__(
        self,
        df,
        cleaner=None,
        flip_rules=False,
    ):
        self.df = df
        self.cleaner = cleaner
        self.flip_rules = flip_rules
        
        if self.flip_rules:
            self.df['rule'] = self.df['rule'].apply(lambda x: flip_rule(x))
        
        if self.cleaner:
            self.df['rule'] = self.df['rule'].apply(lambda x: self.cleaner(x))
            self.df['body'] = self.df['body'].apply(lambda x: self.cleaner(x))
            
        self.positive_examples_d = df[['rule', 'body']][df['rule_violation'] == 1].groupby('rule').agg(list).to_dict()['body']
        self.negative_examples_d = df[['rule', 'body']][df['rule_violation'] == 0].groupby('rule').agg(list).to_dict()['body']
        
        return None
    
    def get_rnd_examples(self, row, n_positive=2, n_negative=2, rnd_examples_position=False):
        rule = row['rule']
        body = row['body']
        
        examples_v = []
        
        while len(examples_v) < n_positive:
            for i_try in range(5):
                example = random.choice( self.positive_examples_d[rule] )
                element = (example, 1)
                if (example != body) and (element not in examples_v):
                    break
                    
            examples_v.append(element)
            
                
        while len(examples_v) < n_negative + n_positive:
            for i_try in range(5):
                example = random.choice( self.negative_examples_d[rule] )
                element = (example, 0)
                if (example != body) and (element not in examples_v):
                    break
                    
            examples_v.append(element)
        
        if rnd_examples_position:
            random.shuffle( examples_v )
            
        return examples_v
    
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        rule = row['rule']
        
        if self.flip_rules:
            sample_d = {
                'anchor': rule,
                'positive': random.choice( self.positive_examples_d[rule] ),
                'negative': random.choice( self.negative_examples_d[rule] ),
            }
            
        else:
            sample_d = {
                'anchor': rule,
                'positive': random.choice( self.negative_examples_d[rule] ),  # Has to be close to the rule
                'negative': random.choice( self.positive_examples_d[rule] ),  # Has to be far form the rule
            }
            
        return sample_d
    
    def __len__(self):
        return len(self.df)
    
    def build_triplet_dataset(self, epochs=1, seed=None):
        random.seed(seed)
        ds_df = {
            'anchor': list(),
            'positive': list(),
            'negative': list(),
        }
        
        for _ in range(epochs):
            for idx in range(self.__len__()):
                for k, v in self[idx].items():
                    ds_df[k].append(v)
            
        dataset = Dataset.from_dict(ds_df)
    
        return dataset

    def get_all_bodies(self):
        return self.df['body'].unique().tolist()
    
    def get_all_rules(self):
        return self.df['rule'].unique().tolist()
    


def load_trn_fit_ds():
    # Train dataset and val dataset
    df_public = pd.read_csv("../input/jigsaw-agile-community-rules/train.csv").sample(frac=1.0, random_state=DS_SEED).reset_index(drop=True)
    
    if PUBLIC_DS_TRN_PROP == 1:
        df_public_trn = df_public.reset_index(drop=True).copy()
        
    else:
        df_public_trn = train_test_split(
            df_public,
            train_size=PUBLIC_DS_TRN_PROP,
            random_state=DS_SEED,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    
    if PUBLIC_DS_FIT_PROP == 1:
        df_public_fit = df_public.reset_index(drop=True).copy()
        
    else:
        df_public_fit = train_test_split(
            df_public,
            train_size=PUBLIC_DS_FIT_PROP,
            random_state=DS_SEED+1,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    df_private_tst = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")

    df_trn = pd.concat(
        [
            create_clf_dataset(df_public_trn),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body']).reset_index(drop=True).copy()
    
    df_fit = pd.concat(
        [
            create_clf_dataset(df_public_fit),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body','rule_violation']).reset_index(drop=True).copy()
    
    
    ds_trn = JigsawDatasetTriplet(
        create_clf_dataset(df_trn),
        cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
    
    ds_fit = JigsawDatasetTriplet(
        create_clf_dataset(df_fit),
        cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
    return ds_trn, ds_fit


if __name__ == '__main__':
    assert os.path.exists(MODEL_NAME)
    ds_trn, ds_fit = load_trn_fit_ds()

    print()
    print('------ CFG ----------')
    print(f'{MODEL_NAME=}')
    print(f'{USE_LORA=}')
    if USE_LORA:
        print(f'{INIT_LORA=}')
    print()
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')
    print()
    print(f'{PUBLIC_DS_TRN_PROP=}')
    print(f'{PUBLIC_DS_FIT_PROP=}')
    print()
    print(f'{len(ds_trn)=}')
    print(f'{len(ds_fit)=}')
    print()


    if False:
        import pandas as pd
        import sys, os
    
        x = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")
        if len(x) <= 10:
            os.system('cp ../input/jigsaw-agile-community-rules/sample_submission.csv ./submission.csv')

## train_margin.py

In [ ]:
%%writefile train_margin.py

from transformers.utils import is_torch_bf16_gpu_available
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from settings_margin import *
import torch


def load_embedding_model(
    model_path,
):
    word_embedding_model = models.Transformer(model_path, max_seq_length=MAX_SEQ_LENGTH, do_lower_case=DO_LOWER_CASE)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    return model
    

def main():
    ds_trn, ds_fit = load_trn_fit_ds()

    print(f'{len(ds_trn)=}')
    
    model = load_embedding_model(
        model_path=MODEL_NAME,
    )
    ds_triplet_trn = ds_trn.build_triplet_dataset(epochs=EPOCHS, seed=DS_SEED)

    loss = TripletLoss(model=model, triplet_margin=TRIPLET_MARGIN)
    
    # Calculate max_steps for small datasets
    NUM_DEVICES = torch.cuda.device_count()
    
    STEPS_PER_EPOCH = max(1, len(ds_triplet_trn) / (EPOCHS * GRAD_ACC_NUM * TRAIN_BS * NUM_DEVICES ))
    MAX_STEPS =  max(1, len(ds_triplet_trn) / (GRAD_ACC_NUM * TRAIN_BS * NUM_DEVICES ))
    LOGGING_STEPS = max(1, STEPS_PER_EPOCH // 8)
    
    STEPS_PER_EPOCH = int(STEPS_PER_EPOCH) if int(STEPS_PER_EPOCH) == STEPS_PER_EPOCH else int(STEPS_PER_EPOCH) + 1
    MAX_STEPS = int(MAX_STEPS) if int(MAX_STEPS) == MAX_STEPS else int(MAX_STEPS) + 1
    
    LOGGING_STEPS = max(1, STEPS_PER_EPOCH // 4)
    
    print()
    print('------ CFG----------')
    print(f'{len(ds_triplet_trn)=}')
    print(f'{LR=}')
    print(f'{TRAIN_BS=}')
    print(f'{EPOCHS=}')
    print(f'{GRAD_ACC_NUM=}')
    print(f'{NUM_DEVICES=}')
    print(f'{STEPS_PER_EPOCH=}')
    print(f'{LOGGING_STEPS=}')
    print(f'{MAX_STEPS=}')
    print(f'{MODEL_OUTPUT_PATH=}')
    print('----------------')
    print()
    
    if os.path.exists(MODEL_OUTPUT_PATH):
        os.system(f'rm -r {MODEL_OUTPUT_PATH}')
    
    if USE_LORA:
        args = SentenceTransformerTrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=1,
            per_device_train_batch_size=TRAIN_BS,
            warmup_steps=0,
            optim="paged_adamw_8bit",  # 8-bit optimizer for memory efficiency
            lr_scheduler_type="linear",
            warmup_ratio=0.1,  # Warm up learning rate over 10% of steps
            learning_rate=LR,
            weight_decay=0.01,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            save_total_limit=0,
            fp16=True,
            max_grad_norm=1.0,
            dataloader_drop_last=False,
            gradient_checkpointing=True,
            gradient_accumulation_steps=GRAD_ACC_NUM,
            #max_steps=MAX_STEPS,
            report_to="none"
        )
        
    else:
        args = SentenceTransformerTrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=1,
            per_device_train_batch_size=TRAIN_BS,
            warmup_steps=0,
            learning_rate=LR,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            save_total_limit=0,
            fp16=True,
            max_grad_norm=1.0,
            dataloader_drop_last=False,
            gradient_checkpointing=True,
            gradient_accumulation_steps=GRAD_ACC_NUM,
            #max_steps=MAX_STEPS,
            report_to="none"
        )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=ds_triplet_trn,
        loss=loss,
    )
    
    trainer.train()
    
    print(f"Saving fine-tuned model to {MODEL_OUTPUT_PATH}...")
    model.save_pretrained(MODEL_OUTPUT_PATH)

    return None


if __name__ == '__main__':
    main()

## embeddings_inference_margin.py

In [ ]:
%%writefile embeddings_inference_margin.py

import os
import random
import argparse
import pandas as pd
import numpy as np
import random
import multiprocessing as mp

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

from settings_margin import *

def fit_gbm_models(X_trn, y_trn, X_tst, n_splits=4, seed=2434, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)

    print(f'{X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'Fitting fold: {i_fold}')

        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1
        }
        if seed:
            params['seed'] = seed + i_fold
            
        callbacks_v = [lgb.early_stopping(stopping_rounds=50)]

        if verbose:
            callbacks_v.append(lgb.log_evaluation(period=50))
            
        # Entrenar
        model = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=callbacks_v
        )

        # Predicciones
        y_fold_pred = model.predict(X_tst, num_iteration=model.best_iteration)
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)

    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v

def fit_and_predict_embeddings(fit_samples_df, tst_samples_df):
    all_tst_preds_v = np.zeros( (len(tst_samples_df), 2) )
    n_clf = 0
    for i_rule, rule in enumerate(tst_samples_df.rule.unique()):
        f_fit = (fit_samples_df['rule'] == rule)
        f_pred = (tst_samples_df.rule.values == rule)

        if False:
            print(f'Fitting: LogisticRegression DS[rule={i_rule}]')
            clf = LogisticRegression(max_iter=1000, C=1)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1

        if True:
            print(f'Fitting: SVC rule: DS[rule={i_rule}]')
            clf = SVC(kernel="linear", C=2, probability=True, random_state=42)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1


        if True:
            print(f'Fitting: LGBM rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_gbm_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1

    if False:
        print('Fitting: LogisticRegression FullDS')
        clf = LogisticRegression(max_iter=1000, C=2)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1

    if False:
        print('Fitting: SVC FullDS')
        clf =SVC(kernel="linear", C=2, probability=True, random_state=42)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1


    if False:
        print('Fitting: LGBM FullDS')
        all_tst_preds_v += fit_gbm_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=4
        )[0]
        n_clf += 1

    all_tst_preds_v = all_tst_preds_v / n_clf

    return all_tst_preds_v



def load_embedding_model(
    model_path="../input/baai/transformers/bge-base-en-v1.5/1",
):
    from sentence_transformers import SentenceTransformer, models
    word_embedding_model = models.Transformer(model_path, max_seq_length=MAX_SEQ_LENGTH, do_lower_case=DO_LOWER_CASE)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    return model

def generate_embeddings_d(texts_v, model, batch_size=EVAL_BS):
    embeddigs_v = model.encode(
        sentences=texts_v,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    return embeddigs_v

def run_inference_on_device(texts_v, model_path):
    model = load_embedding_model(
        model_path=model_path,
    )
    embeddigs_v = generate_embeddings_d(texts_v, model)
    
    text2embed_d = {
        t : e
        for t, e in 
        zip(texts_v, embeddigs_v)
    }
    return text2embed_d
    

def worker(device_id, texts_v, model_path, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] 🔥 Running on GPU {device_id}, {len(texts_v)=} {model_path=}")
    
    text2embed_d = run_inference_on_device(texts_v, model_path)
    
    return_dict[f'text2embed_d_slice{device_id}'] = text2embed_d
    print(f"[Worker {device_id}] ✅ Inference OK!")
    
    
def create_centroids(ds, text2embed_d, rule2embed_d, flip_rules=False):
    rule_centroids = {}
    for rule in ds.get_all_rules():
        neg_embeddings = []
        pos_embeddings = []
        if flip_rules:
            for body in ds.negative_examples_d[rule]:
                neg_embeddings.append(
                    text2embed_d[body]
                )


            for body in ds.positive_examples_d[rule]:
                pos_embeddings.append(
                    text2embed_d[body]
                )
                
                
        else:
            for body in ds.positive_examples_d[rule]:
                neg_embeddings.append(
                    text2embed_d[body]
                )


            for body in ds.negative_examples_d[rule]:
                pos_embeddings.append(
                    text2embed_d[body]
                )

        pos_embeddings = np.array(pos_embeddings)
        neg_embeddings = np.array(neg_embeddings)

        # Compute mean centroids
        pos_centroid = pos_embeddings.mean(axis=0)
        neg_centroid = neg_embeddings.mean(axis=0)

        # Normalize centroids
        pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
        neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)


        rule_centroids[rule] = {
            'positive': pos_centroid,
            'negative': neg_centroid,
            'pos_count': len(pos_embeddings),
            'neg_count': len(neg_embeddings),
            'rule_embedding': rule2embed_d[rule]
        }
        
    return rule_centroids


def predict_test_df(df_tst, rule_centroids_d, flip_rules=False):
    predictions = []
    df_tst['scores'] = -1.0
    
    for rule in df_tst['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        f_rule = (df_tst['rule'] == rule)

        query_embeddings = df_tst.loc[f_rule, 'body_embeddings'].values.tolist()
        query_embeddings = np.vstack(query_embeddings)
        
        assert len(query_embeddings) > 0
        
        pos_centroid = rule_centroids_d[rule]['positive']
        neg_centroid = rule_centroids_d[rule]['negative']

        # Compute Euclidean distances
        pos_distances = np.linalg.norm(query_embeddings - pos_centroid, axis=1)  # pos means agree with the rule
        neg_distances = np.linalg.norm(query_embeddings - neg_centroid, axis=1)  # neg rule violation

        if flip_rules:
            rule_scores = neg_distances - pos_distances
        else:
            rule_scores = pos_distances - neg_distances
        
        df_tst.loc[f_rule, 'scores'] = rule_scores

    print(f"Made predictions for {len(predictions)} test examples")
    
    assert (df_tst['scores'] != -1).any()
    
    if False:
        scores = df_tst['scores'].values
        df_tst['rule_violation'] =  (scores - scores.min()) / (scores.max() - scores.min())
    else:    
        df_tst['rule_violation'] =  (df_tst['scores'].rank() -1) / len(df_tst['scores'])
        
    return None

def main():
    parser = argparse.ArgumentParser(description="Margin model inference")
    parser.add_argument("--model-path", default=MODEL_OUTPUT_PATH, type=str)
    parser.add_argument("--inference-dataset", default="../input/jigsaw-agile-community-rules/test.csv", type=str)
    parser.add_argument("--output-csv", type=str, default='./submission.csv')
    args = parser.parse_args()

    model_path = args.model_path
    inference_dataset = args.inference_dataset
    output_csv = args.output_csv

    print('Inference settings:')
    print(f'{model_path=}')
    print(f'{inference_dataset=}')
    print(f'{output_csv=}')
    print()
    
    manager = mp.Manager()
    return_dict = manager.dict()
    
    _, ds_fit = load_trn_fit_ds()

    df_tst = pd.read_csv(inference_dataset)
    
    if FLIP_RULES:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: flip_rule(x))
    
    if USE_CLEANER:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: cleaner(x))
        df_tst['body'] = df_tst['body'].apply(lambda x: cleaner(x))
            
    
    all_rules_v = sorted(
        set(
            ds_fit.get_all_rules() + df_tst['rule'].unique().tolist()
        )
    )
    
    # Embeddings inference:
    all_texts_v = sorted(
        set(
            ds_fit.get_all_bodies() + df_tst['body'].unique().tolist() + all_rules_v
        )
    )
    
    print('😱 Starting Embeddings Inference:')

    text_mid = len(all_texts_v) // 2

    p0 = mp.Process(
        target=worker,
        args=(
            0, # Device
            all_texts_v[:text_mid], # texts_v
            model_path,
            return_dict,
        )
    )
    p1 = mp.Process(
        target=worker,
        args=(
            1, # Device
            all_texts_v[text_mid:], # texts_v
            model_path,
            return_dict,
        )
    )
    p0.start()
    p1.start()
    p0.join()
    p1.join()
    
    all_texts2embed_d = {}    
    all_texts2embed_d.update( return_dict['text2embed_d_slice0'] )
    all_texts2embed_d.update( return_dict['text2embed_d_slice1'] )

    ds_fit.df['body_embeddings'] = ds_fit.df['body'].apply(lambda t: all_texts2embed_d[t])
    df_tst['body_embeddings'] = df_tst['body'].apply(lambda t: all_texts2embed_d[t])
    rule2embed_d = {r:all_texts2embed_d[r] for r in all_rules_v}
    
    rule_centroids_d = create_centroids(ds_fit, all_texts2embed_d, rule2embed_d, flip_rules=FLIP_RULES)
    
    if not CLF_OUTPUT:
        predict_test_df(df_tst, rule_centroids_d, flip_rules=FLIP_RULES)
        
    else:
        def calc_embeddings(row):
            if CLF_EMBEDDINGS_TYPE == 0:
                embedings = np.concatenate(
                    [
                        row.body_embeddings - rule_centroids_d[row.rule]['positive'],
                        row.body_embeddings - rule_centroids_d[row.rule]['negative'],
                    ],
                    axis=0
                )
            elif CLF_EMBEDDINGS_TYPE == 1:
                embedings = row.body_embeddings
            elif CLF_EMBEDDINGS_TYPE == 2:
                embedings = row.body_embeddings - rule_centroids_d[row.rule]['positive'] + rule_centroids_d[row.rule]['negative']
                
            elif CLF_EMBEDDINGS_TYPE == 3:
                embedings = np.concatenate(
                    [
                        row.body_embeddings,
                        row.body_embeddings - rule_centroids_d[row.rule]['positive'],
                        row.body_embeddings - rule_centroids_d[row.rule]['negative'],
                    ],
                    axis=0
                )
                
            else:
                raise NotImplementedError(f'{CLF_EMBEDDINGS_TYPE=} ???')
            return embedings
        
        ds_fit.df['embeddings'] = ds_fit.df.apply(calc_embeddings, axis=1)
        df_tst['embeddings'] = df_tst.apply(calc_embeddings, axis=1)

        print('☢️ Fitting embeddings CLFs.')
        all_tst_preds_v = fit_and_predict_embeddings(ds_fit.df, df_tst)
        df_tst['rule_violation'] = all_tst_preds_v[:, 1]
    
    
    if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
        from sklearn.metrics import accuracy_score, roc_auc_score
        df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
        AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=df_tst.rule_violation.values)
        ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=df_tst.rule_violation.values > 0.5)

        print('Test GT Found, computing Final score:')
        print(f'-Test {AUC=:0.4f}')
        print(f'-Test {ACC=:0.4f}')
    
        save_obj(
            {
                'all_rules_v': all_rules_v,
                'all_texts2embed_d': all_texts2embed_d,
                'df_tst': df_tst,
                'ds_fit': ds_fit,
                'rule_centroids_d': rule_centroids_d,
            },
            './embeddings.pickle'
        )

    df_tst[['row_id', 'rule_violation']].to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")

    return None

if __name__ == "__main__":
    main()

## constants_margin.py: FLIP_RULES

In [ ]:
%%writefile constants_margin.py


# Main configuration parameters
#MODEL_NAME = '../input/qwen-3-embedding/transformers/0.6b/1'
MODEL_NAME = '../input/baai/transformers/bge-base-en-v1.5/1'
#MODEL_NAME = '../input/baai/transformers/bge-reranker-large/1'
#MODEL_NAME = '../input/test-finetuned-bge-trn-2e'
INIT_LORA = None

USE_LORA = False
TEST = 40
DS_SEED = 42 + TEST
EPOCHS = 3 # Training epochs
FLIP_RULES = True

MAX_SEQ_LENGTH = 128
DO_LOWER_CASE = False
USE_CLEANER = False

CLF_OUTPUT = True
CLF_EMBEDDINGS_TYPE = 0
TRIPLET_MARGIN = 0.4 #0.25
LR = 1e-4

PUBLIC_DS_TRN_PROP = 1.00
PUBLIC_DS_FIT_PROP = 1.00

TRAIN_BS = 32
EVAL_BS = 64
GRAD_ACC_NUM = 1

OUTPUT_DIR = f"./margin_embed_output"
MODEL_OUTPUT_PATH = f"{OUTPUT_DIR}/best_model"


## RUN: FLIP_RULES

In [ ]:
!python settings_margin.py

In [ ]:
!python train_margin.py

In [ ]:
!ls margin_embed_output/best_model

In [ ]:
!python embeddings_inference_margin.py --inference-dataset /kaggle/input/jigsaw-agile-community-rules/test.csv --output-csv /kaggle/working/submission_triplet_ttt_RF.csv

## constants_margin.py: noFLIP_RULES

In [ ]:
%%writefile constants_margin.py


# Main configuration parameters
#MODEL_NAME = '../input/qwen-3-embedding/transformers/0.6b/1'
MODEL_NAME = '../input/baai/transformers/bge-base-en-v1.5/1'
#MODEL_NAME = '../input/baai/transformers/bge-reranker-large/1'
#MODEL_NAME = '../input/test-finetuned-bge-trn-2e'
INIT_LORA = None

USE_LORA = False
TEST = 43
DS_SEED = 42 + TEST
EPOCHS = 3 # Training epochs
FLIP_RULES = False

MAX_SEQ_LENGTH = 128
DO_LOWER_CASE = False
USE_CLEANER = False

CLF_OUTPUT = True
CLF_EMBEDDINGS_TYPE = 0
TRIPLET_MARGIN = 0.4 #0.25
LR = 1e-4

PUBLIC_DS_TRN_PROP = 1.00
PUBLIC_DS_FIT_PROP = 1.00

TRAIN_BS = 32
EVAL_BS = 64
GRAD_ACC_NUM = 1

OUTPUT_DIR = f"./margin_embed_output"
MODEL_OUTPUT_PATH = f"{OUTPUT_DIR}/best_model"

## RUN: noFLIP_RULES

In [ ]:
!python settings_margin.py

In [ ]:
!python train_margin.py

In [ ]:
!python embeddings_inference_margin.py --inference-dataset /kaggle/input/jigsaw-agile-community-rules/test.csv --output-csv /kaggle/working/submission_triplet_ttt_noRF.csv

# ArcFace

## settings_clf.py

In [ ]:
%%writefile settings_clf.py

import os
import re
import pickle
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
from urllib.parse import urlparse

# Main configuration parameters
MODEL_NAME = '../input/qwen-3-embedding/transformers/0.6b/1'
#MODEL_NAME = '../input/baai/transformers/bge-base-en-v1.5/1'
#MODEL_NAME = '../input/baai/transformers/bge-reranker-large/1'

#MODEL_NAME = '../input/test-finetuned-bge-trn-2e'
INIT_LORA = None

LORA_RANK = 16
LORA_ALPHA = 16

N_EVALS_PER_EPOCH = 5

CALC_INIT_ARC_WEIGHTS = True

TEST = 46
DS_SEED = 42 + TEST
SEED = 38209
EPOCHS = 1 # Training epochs
FLIP_RULES = False

MAX_SEQ_LENGTH = 200
DO_LOWER_CASE = False
USE_CLEANER = True

ARCFACE_MARGIN = 0.4
ARCFACE_SCALE = 32.0
LR = 1e-4

PUBLIC_DS_TRN_PROP = 1.00
PUBLIC_DS_FIT_PROP = 1.00

TRAIN_BS = 32
EVAL_BS = 64
GRAD_ACC_NUM = 1

OUTPUT_DIR = f"./argface_embed_output"
MODEL_OUTPUT_PATH = f"{OUTPUT_DIR}/best_model"     


def save_obj(obj, filename, verbose=True):
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)

    if verbose:
        print(f'Saved: {filename}')

    return None

def load_obj(filename, verbose=True):
    with open(filename, 'rb') as f:
        obj = pickle.load(f)

    if verbose:
        print(f'Loaded: {filename}')

    return obj


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))


def flip_rule(rule):
    str_replacement_v = [
        ("don[' ]t", ''),
        ('do not', ''),
        
        ('are not', ''),
        ("aren[ ']t", ''),
        
        ('is not', ''),
        ("isn[ ']t", ''),
        
        ('no', ''),
        (' +', ' '),
    ]
    
    for f, r in str_replacement_v:
        rule = re.sub(f, r, rule.lower())
        
    return rule.strip()
    

def create_clf_dataset(df):
    dfs_to_concat_v = []
    if ('positive_example_1' in df.columns) and ('positive_example_2' in df.columns):
        all_positives_df = pd.concat(
            [
                df[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'}),
                df[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_positives_df['rule_violation'] = 1
        dfs_to_concat_v.append(all_positives_df)
    
    
    if ('negative_example_1' in df.columns) and ('negative_example_2' in df.columns):
        all_negatives_df = pd.concat(
            [
                df[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'}),
                df[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_negatives_df['rule_violation'] = 0
        dfs_to_concat_v.append(all_negatives_df)
    
    
    if ('body' in df.columns) and ('rule_violation' in df.columns):
        all_bodies_df = df[['rule', 'body', 'rule_violation']].copy()
        dfs_to_concat_v.append(all_bodies_df)
        
        
    if len(dfs_to_concat_v) > 1:
        all_examples_df = pd.concat(
            dfs_to_concat_v,
            axis=0
        ).drop_duplicates()
        
    elif len(dfs_to_concat_v) == 1:
        all_examples_df = dfs_to_concat_v[0].drop_duplicates()
        
    else:
        raise Exception('No data found')
        
    return all_examples_df.reset_index(drop=True)


class JigsawDatasetCLF():
    def __init__(
        self,
        df,
        tokenizer,
        cleaner=None,
        flip_rules=False,
    ):
        self.df = df.copy()
        self.tokenizer = tokenizer
        self.cleaner = cleaner
        self.flip_rules = flip_rules
        
        if self.flip_rules:
            self.df['rule'] = self.df['rule'].apply(lambda x: flip_rule(x))
        
        if self.cleaner:
            self.df['rule'] = self.df['rule'].apply(lambda x: self.cleaner(x))
            self.df['body'] = self.df['body'].apply(lambda x: self.cleaner(x))
            
        self.all_rules_v = sorted(self.df['rule'].unique())
        
        if 'rule_violation' in df.columns:
            self.positive_examples_d = df[['rule', 'body']][df['rule_violation'] == 1].groupby('rule').agg(list).to_dict()['body']
            self.negative_examples_d = df[['rule', 'body']][df['rule_violation'] == 0].groupby('rule').agg(list).to_dict()['body']
            
            self.df['label'] = self.df['rule'].apply(lambda r: self.all_rules_v.index(r) + 1)
            self.df.loc[df['rule_violation'] == 0, 'label'] = 0
        
        return None
    
    
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        #sample_d = self.tokenizer(row['rule'] + ': ' + row['body'])
        sample_d = self.tokenizer(row['body'])
        
        if 'label' in row:
            sample_d['labels'] =  row['label']
            
        return sample_d
    
    def __len__(self):
        return len(self.df)

    def get_all_bodies(self):
        return self.df['body'].unique().tolist()
    
    def get_all_rules(self):
        return self.all_rules_v
    


def load_trn_fit_df():
    # Train dataset and val dataset
    df_public = pd.read_csv("../input/jigsaw-agile-community-rules/train.csv").sample(frac=1.0, random_state=DS_SEED).reset_index(drop=True)
    
    if PUBLIC_DS_TRN_PROP == 1:
        df_public_trn = df_public.reset_index(drop=True).copy()
        
    else:
        df_public_trn = train_test_split(
            df_public,
            train_size=PUBLIC_DS_TRN_PROP,
            random_state=DS_SEED,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    
    if PUBLIC_DS_FIT_PROP == 1:
        df_public_fit = df_public.reset_index(drop=True).copy()
        
    else:
        df_public_fit = train_test_split(
            df_public,
            train_size=PUBLIC_DS_FIT_PROP,
            random_state=DS_SEED+1,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    df_private_tst = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")

    df_trn = pd.concat(
        [
            create_clf_dataset(df_public_trn),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body']).reset_index(drop=True).copy()
    
    df_fit = pd.concat(
        [
            create_clf_dataset(df_public_fit),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body','rule_violation']).reset_index(drop=True).copy()

    return df_trn, df_fit


if __name__ == '__main__':
    assert os.path.exists(MODEL_NAME)
    df_trn, df_fit = load_trn_fit_df()

    print()
    print('------ CFG ----------')
    print(f'{MODEL_NAME=}')

    print()
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')
    print()
    print(f'{PUBLIC_DS_TRN_PROP=}')
    print(f'{PUBLIC_DS_FIT_PROP=}')
    print()
    print(f'{len(df_trn)=}')
    print(f'{len(df_fit)=}')
    print()

    if False:
        import pandas as pd
        import sys, os
    
        x = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")
        if len(x) <= 10:
            os.system('cp ../input/jigsaw-agile-community-rules/sample_submission.csv ./submission.csv')


## train_clf.py

In [ ]:
%%writefile train_clf.py


import math
from tqdm import tqdm
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from settings_clf import *


def mean_pooling(last_hidden_state, attention_mask):
    token_embeddings = last_hidden_state  # [B, L, H]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        # normalizar
        embeddings = embeddings.to(torch.float32)
        weights = self.weight.to(torch.float32)
        
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)

        cosine = F.linear(embeddings, weights)
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.m)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        output *= self.s
        logits = self.s * cosine
        loss = F.cross_entropy(output, labels)
        return loss, logits
    
    def project(self, embeddings):
        # weights norm
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, weights)
        logits = self.s * cosine
        return logits



class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5, eps=1e-7, reduce_class0=True):
        super().__init__()
        self.s = s
        self.m = m
        self.eps = eps
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        if reduce_class0:
            w = torch.ones(out_features)
            w[0] = 1.0 / (out_features - 1)
            self.register_buffer("class_weights", w)
        else:
            self.class_weights = None

    def forward(self, embeddings, labels):
        # 🔹 estabilidad: trabajar siempre en float32
        embeddings = embeddings.to(torch.float32)
        W = self.weight.to(torch.float32)

        embeddings = F.normalize(embeddings, p=2, dim=1)
        W = F.normalize(W, p=2, dim=1)

        cosine = F.linear(embeddings, W)
        cosine = torch.clamp(cosine, -1.0 + self.eps, 1.0 - self.eps)

        sine = torch.sqrt(torch.clamp(1.0 - cosine ** 2, min=0.0))

        phi = cosine * self.cos_m - sine * self.sin_m

        one_hot = torch.zeros_like(cosine, device=embeddings.device)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        logits *= self.s

        loss = F.cross_entropy(
            logits,
            labels,
            weight=self.class_weights.to(embeddings.device) if self.class_weights is not None else None,
        )
        return loss, logits

    def project(self, embeddings):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        W = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, W)
        return self.s * cosine

class QwenEmbeddingModel(nn.Module):
    def __init__(self, model, arcface_loss, pooling_fn):
        super().__init__()
        self.model = model
        self.arcface_loss = arcface_loss
        self.pooling_fn = pooling_fn

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.pooling_fn(outputs.last_hidden_state, attention_mask)
        
        if labels is not None:
            loss, logits = self.arcface_loss(embeddings, labels)
            return {"loss": loss, "embeddings": embeddings, "logits": logits}
        
        logits = self.arcface_loss.project(embeddings)
        
        return {"embeddings": embeddings, "logits": logits}
    
    def gradient_checkpointing_enable(self, **kwargs):
        if hasattr(self.model, "gradient_checkpointing_enable"):
            self.model.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        if hasattr(self.model, "gradient_checkpointing_disable"):
            self.model.gradient_checkpointing_disable()
    
    def save_model(self, outputs_dir):
        self.model.save_pretrained(outputs_dir)
        
        save_obj(
            self.arcface_loss.state_dict(),
            os.path.join(outputs_dir, 'arcface_weights.pickle'),
        )
        
        return None
    
    
    def encode(
        self, 
        ds,
        data_collator,
        batch_size: int = 32, 
        normalize: bool = True,
        output_numpy: bool = False,
        verbose: bool = True,
    ):

        self.model.eval()

        embeddings_v = []
        with torch.no_grad():
            itt = DataLoader(ds, batch_size=batch_size, collate_fn=data_collator)
            if verbose:
                itt = tqdm(itt)
                
            for data in itt:
                data = data.to(self.model.device)
                with torch.autocast(device_type="cuda", dtype=None):
                    outputs = self.model(**data)
                    
                embeddings = self.pooling_fn(
                    last_hidden_state=outputs.last_hidden_state.to(torch.float32),
                    attention_mask=data["attention_mask"]
                )
                
                if normalize:
                    embeddings = F.normalize(embeddings, p=2, dim=1)

                embeddings_v.append(embeddings.cpu())

            embeddings_v = torch.cat(embeddings_v, dim=0)
        
            if output_numpy:
                embeddings_v = embeddings_v.numpy()
            
        return embeddings_v
    
    
def load_base_model(model_name, init_lora=None, is_trainable=True):
    from peft import get_peft_model, LoraConfig, TaskType, PeftModel
    from transformers import AutoModel, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)
    
    torch.manual_seed(SEED)
    
    if init_lora is None:
        # Configure LoRA parameters
        lora_config = LoraConfig(
            r=LORA_RANK,  # Rank of the update matrices
            lora_alpha=LORA_ALPHA,  # Alpha parameter for LoRA scaling
            lora_dropout=0.05,  # Dropout probability for LoRA layers
            task_type=TaskType.FEATURE_EXTRACTION,
            bias='none',  # Don't train bias terms
            # Target the attention and MLP modules of the transformer
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj",
            ]
        )

        model = get_peft_model(base_model, lora_config)
    
    
    else:
        model = PeftModel.from_pretrained(
            base_model,
            init_lora,
            is_trainable=is_trainable,
        )
    
    return model, tokenizer


def load_embedding_model(base_model, init_lora, num_classes=2, load_arc_weights=False, is_trainable=True):
    from transformers import DataCollatorWithPadding
    
    model, tokenizer = load_base_model(base_model, init_lora=init_lora, is_trainable=is_trainable)
    
    # Initial Model
    arcface_loss = ArcFaceLoss(
        in_features=model.config.hidden_size,
        out_features=num_classes,
        s=ARCFACE_SCALE,
        m=ARCFACE_MARGIN,
    )
    
    if load_arc_weights:
        sd = load_obj(
            os.path.join(init_lora, 'arcface_weights.pickle'),
        )
        
        arcface_loss.load_state_dict(sd)
    
    qwen_embeddings_model = QwenEmbeddingModel(model, arcface_loss, mean_pooling)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    
    return qwen_embeddings_model, tokenizer, data_collator


def run_inference_on_device(clf_df, base_model, init_lora):
    qwen_embeddings_model, tokenizer, data_collator = load_embedding_model(base_model, init_lora, num_classes=2, load_arc_weights=False, is_trainable=False)
    
    qwen_embeddings_model.to('cuda:0')
    
    ds = JigsawDatasetCLF(
        clf_df,
        tokenizer=tokenizer,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
     
    embeddings_v = qwen_embeddings_model.encode(
        ds,
        data_collator,
        batch_size=EVAL_BS, 
        normalize=True,
        output_numpy=False,
        verbose=True,
    )
    proc_df = ds.df.copy()
    return proc_df, embeddings_v
    

def worker(device_id, clf_df, base_model, init_lora, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] 🔥 Running on GPU {device_id}, {len(clf_df)=} {init_lora=}")
    
    proc_df, embeddings_v = run_inference_on_device(clf_df, base_model, init_lora)

    return_dict[f'proc_df_slice{device_id}'] = proc_df
    return_dict[f'embeddings_v_slice{device_id}'] = embeddings_v
    print(f"[Worker {device_id}] ✅ Inference OK!")
    
    
def eval_model_embeddings(clf_df, base_model, init_lora):
    import multiprocessing as mp
    
    print('😱 Starting Embeddings Inference:')

    rows_mid = len(clf_df) // 2

    manager = mp.Manager()
    return_dict = manager.dict()
    
    p0 = mp.Process(
        target=worker,
        args=(
            0, # Device
            clf_df.iloc[:rows_mid],
            base_model,
            init_lora,
            return_dict,
        )
    )
    p1 = mp.Process(
        target=worker,
        args=(
            1, # Device
            clf_df.iloc[rows_mid:],
            base_model,
            init_lora,
            return_dict,
        )
    )
    p0.start()
    p1.start()
    p0.join()
    p1.join()
    
    embeddings_v = torch.concat(
        [
            return_dict['embeddings_v_slice0'],
            return_dict['embeddings_v_slice1'],
        ]
    )
    proc_df = pd.concat(
        [
            return_dict['proc_df_slice0'],
            return_dict['proc_df_slice1'],
        ]
    )
    
    return proc_df, embeddings_v


    
def main():
    df_trn, df_fit = load_trn_fit_df()
    
    ds_trn = JigsawDatasetCLF(
        create_clf_dataset(df_trn),
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
    
    print(f'{len(ds_trn)=}')

    if CALC_INIT_ARC_WEIGHTS:
        print('Initial trn embeddings eval')
        #trn_proc_df, trn_embeddings_v = eval_model_embeddings(ds_trn.df[ds_trn.df.label!=0].copy(), base_model=MODEL_NAME, init_lora=INIT_LORA)
        trn_proc_df, trn_embeddings_v = eval_model_embeddings(ds_trn.df.copy(), base_model=MODEL_NAME, init_lora=INIT_LORA)



    from transformers import Trainer, TrainingArguments
    from transformers.utils import is_torch_bf16_gpu_available

    NUM_CLASSES = len(ds_trn.get_all_rules()) + 1

    qwen_embeddings_model, tokenizer, data_collator = load_embedding_model(
        base_model=MODEL_NAME,
        init_lora=INIT_LORA,
        num_classes=NUM_CLASSES,
        load_arc_weights=False,
        is_trainable=True,
    )
    ds_trn.tokenizer = tokenizer
    ds_val = None
        
    if CALC_INIT_ARC_WEIGHTS:
        for label in sorted(trn_proc_df['label'].unique().tolist()):
            f = (trn_proc_df['label'] == label).values
            w = F.normalize(trn_embeddings_v[f].mean(axis=0), dim=0, p=2)
            w = w.type(qwen_embeddings_model.arcface_loss.weight.data.dtype).to(qwen_embeddings_model.arcface_loss.weight.data.device)
            
            if label == 0:
                qwen_embeddings_model.arcface_loss.weight.data[label,:] = 0.5 * qwen_embeddings_model.arcface_loss.weight.data[label,:] + w
                
            else:
                qwen_embeddings_model.arcface_loss.weight.data[label,:] = w
    
    
    # Calculate max_steps for small datasets
    NUM_DEVICES = torch.cuda.device_count()
    
    if os.path.exists(MODEL_OUTPUT_PATH):
        os.system(f'rm -r {MODEL_OUTPUT_PATH}')
    
    print('qwen_embeddings_model, print_trainable_parameters:')
    qwen_embeddings_model.model.print_trainable_parameters()
    
    LOGGING_STRATEGY = ['epoch', 'steps'][1]
    if ds_val is None:
        SAVE_STRATEGY = 'no'
        EVAL_STRATEGY = 'no'
    else:
        SAVE_STRATEGY = LOGGING_STRATEGY
        EVAL_STRATEGY = LOGGING_STRATEGY


    LOGGING_STEPS = len(ds_trn) // (TRAIN_BS * GRAD_ACC_NUM * N_EVALS_PER_EPOCH * torch.cuda.device_count())
    SAVE_TOTAL_LIMIT = N_EVALS_PER_EPOCH + 1

    SAVE_STEPS = LOGGING_STEPS

    print('Training ...')
    print()
    print('------ CFG----------')
    print(f'{NUM_DEVICES=}')
    print(f'{NUM_CLASSES=}')
    print(f'{EPOCHS=}')
    print(f'{EVAL_STRATEGY=}')
    print(f'{LOGGING_STEPS=}')
    print(f'{SAVE_STEPS=}')
    print(f'{SAVE_TOTAL_LIMIT=}')
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')

    print(f'{len(ds_trn)=}')
    if ds_val is None:
        print(f'{ds_val=}')
    else:
        print(f'{len(ds_val)=}')
    print('----------------')
    print()

    # Set up training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        
        num_train_epochs=EPOCHS,
        optim="paged_adamw_8bit",  # 8-bit optimizer for memory efficiency
        #optim="adamw_torch",  # 8-bit optimizer for memory efficiency
        lr_scheduler_type="linear",
        warmup_ratio=0.1,  # Warm up learning rate over 10% of steps
        learning_rate=LR,
        weight_decay=0.01,
        #max_grad_norm=1.0,
        
        # Use BF16 if available, otherwise FP16
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),

        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACC_NUM,
        gradient_checkpointing=True,  # Save memory with gradient checkpointing
        gradient_checkpointing_kwargs={"use_reentrant": False},
        group_by_length=False,
        seed=SEED,
        remove_unused_columns=False,  # Keep all columns in the dataset
        
        report_to="tensorboard",
        metric_for_best_model="AUC",
        greater_is_better=True,

        load_best_model_at_end=(ds_val is not None),
        save_total_limit=SAVE_TOTAL_LIMIT,

        logging_strategy=LOGGING_STRATEGY,
        eval_strategy=EVAL_STRATEGY,
        save_strategy=SAVE_STRATEGY,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
    )

    # Initialize trainer
    trainer = Trainer(
        qwen_embeddings_model,
        args=training_args,
        train_dataset=ds_trn,
        eval_dataset=ds_val,
        data_collator=data_collator,
    )
    
    trainer.train()
    
    print(f"Saving fine-tuned model to {MODEL_OUTPUT_PATH}...")
    qwen_embeddings_model.save_model(MODEL_OUTPUT_PATH)

    return None


if __name__ == '__main__':
    main()

## embeddings_inference_clf.py

In [ ]:
%%writefile embeddings_inference_clf.py

import os
import random
import argparse
import pandas as pd
import numpy as np
import random
import multiprocessing as mp

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

from settings_clf import *
from train_clf import eval_model_embeddings

def fit_gbm_models(X_trn, y_trn, X_tst, n_splits=4, seed=2434, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)

    print(f'{X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'Fitting fold: {i_fold}')

        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1
        }
        if seed:
            params['seed'] = seed + i_fold
            
        callbacks_v = [lgb.early_stopping(stopping_rounds=50)]

        if verbose:
            callbacks_v.append(lgb.log_evaluation(period=50))
            
        # Entrenar
        model = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=callbacks_v
        )

        # Predicciones
        y_fold_pred = model.predict(X_tst, num_iteration=model.best_iteration)
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)

    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v

def fit_and_predict_embeddings(fit_samples_df, tst_samples_df):
    all_tst_preds_v = np.zeros( (len(tst_samples_df), 2) )
    n_clf = 0
    for i_rule, rule in enumerate(tst_samples_df.rule.unique()):
        f_fit = (fit_samples_df['rule'] == rule)
        f_pred = (tst_samples_df.rule.values == rule)

        if False:
            print(f'Fitting: LogisticRegression DS[rule={i_rule}]')
            clf = LogisticRegression(max_iter=1000, C=1)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1

        if False:
            print(f'Fitting: SVC rule: DS[rule={i_rule}]')
            clf = SVC(kernel="linear", C=2, probability=True, random_state=42)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1


        if True:
            print(f'Fitting: LGBM rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_gbm_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1

    if False:
        print('Fitting: LogisticRegression FullDS')
        clf = LogisticRegression(max_iter=1000, C=2)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1

    if False:
        print('Fitting: SVC FullDS')
        clf =SVC(kernel="linear", C=2, probability=True, random_state=42)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1


    if False:
        print('Fitting: LGBM FullDS')
        all_tst_preds_v += fit_gbm_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=4
        )[0]
        n_clf += 1

    all_tst_preds_v = all_tst_preds_v / n_clf

    return all_tst_preds_v


def main():
    parser = argparse.ArgumentParser(description="Margin model inference")
    parser.add_argument("--base-model-path", default=MODEL_NAME, type=str)
    parser.add_argument("--lora-model-path", default=MODEL_OUTPUT_PATH, type=str)
    parser.add_argument("--inference-dataset", default="../input/jigsaw-agile-community-rules/test.csv", type=str)
    parser.add_argument("--output-csv", type=str, default='./submission.csv')
    args = parser.parse_args()

    base_model_path = args.base_model_path
    lora_model_path = args.lora_model_path
    inference_dataset = args.inference_dataset
    output_csv = args.output_csv

    print('Inference settings:')
    print(f'{base_model_path=}')
    print(f'{lora_model_path=}')
    print(f'{inference_dataset=}')
    print(f'{output_csv=}')
    print()
    
    df_tst = pd.read_csv(inference_dataset)
    
    if FLIP_RULES:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: flip_rule(x))
    
    if USE_CLEANER:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: cleaner(x))
        df_tst['body'] = df_tst['body'].apply(lambda x: cleaner(x))
        
    _, df_fit = load_trn_fit_df()
    
    ds_fit = JigsawDatasetCLF(
        create_clf_dataset(df_fit),
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
    
    ds_tst = JigsawDatasetCLF(
        df_tst,
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
    )
    
    embeddings_df = pd.concat(
        [
            ds_fit.df[['rule', 'body']],
            ds_tst.df[['rule', 'body']],
        ],
        axis=0
    ).drop_duplicates().reset_index(drop=True)
    
    proc_df, embeddings_v = eval_model_embeddings(embeddings_df, base_model=base_model_path, init_lora=lora_model_path)
    
    all_texts2embed_d = {}
    for body, embedding in zip(proc_df.body.values,embeddings_v):
        all_texts2embed_d[body] = embedding
    
    ds_fit.df['embeddings'] = ds_fit.df['body'].apply(lambda t: all_texts2embed_d[t])
    df_tst['embeddings'] = df_tst['body'].apply(lambda t: all_texts2embed_d[t])
    
    print('☢️ Fitting embeddings CLFs.')
    all_tst_preds_v = fit_and_predict_embeddings(ds_fit.df, df_tst)
    df_tst['rule_violation'] = all_tst_preds_v[:, 1]
    
    
    if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
        from sklearn.metrics import accuracy_score, roc_auc_score
        df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
        AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=df_tst.rule_violation.values)
        ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=df_tst.rule_violation.values > 0.5)

        print('Test GT Found, computing Final score:')
        print(f'-Test {AUC=:0.4f}')
        print(f'-Test {ACC=:0.4f}')

        if False:
            save_obj(
                {
                    'all_texts2embed_d': all_texts2embed_d,
                    'df_tst': df_tst,
                    'ds_fit': ds_fit,
                },
                './embeddings.pickle'
            )

    df_tst[['row_id', 'rule_violation']].to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")

    return None

if __name__ == "__main__":
    main()

## RUN

In [ ]:
!python settings_clf.py

In [ ]:
!python train_clf.py

In [ ]:
!python embeddings_inference_clf.py --inference-dataset /kaggle/input/jigsaw-agile-community-rules/test.csv --output-csv /kaggle/working/submission_arcface_ttt.csv

# Deberta CLFv3 inference

## settings_clf.py

In [ ]:
%%writefile settings_clf.py

import os
import re
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from urllib.parse import urlparse

from constants_clf import *  


def save_obj(obj, filename, verbose=True):
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)

    if verbose:
        print(f'Saved: {filename}')

    return None

def load_obj(filename, verbose=True):
    with open(filename, 'rb') as f:
        obj = pickle.load(f)

    if verbose:
        print(f'Loaded: {filename}')

    return obj


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text)).strip()


def flip_rule(rule):
    str_replacement_v = [
        ("don[' ]t", ''),
        ('do not', ''),
        
        ('are not', ''),
        ("aren[ ']t", ''),
        
        ('is not', ''),
        ("isn[ ']t", ''),
        
        ('no', ''),
        (' +', ' '),
    ]
    
    for f, r in str_replacement_v:
        rule = re.sub(f, r, rule.lower())
        
    return rule.strip()
    

def create_clf_dataset(df):
    dfs_to_concat_v = []
    if ('positive_example_1' in df.columns) and ('positive_example_2' in df.columns):
        all_positives_df = pd.concat(
            [
                df[['rule', 'positive_example_1']].rename(columns={'positive_example_1': 'body'}),
                df[['rule', 'positive_example_2']].rename(columns={'positive_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_positives_df['rule_violation'] = 1
        dfs_to_concat_v.append(all_positives_df)
    
    
    if ('negative_example_1' in df.columns) and ('negative_example_2' in df.columns):
        all_negatives_df = pd.concat(
            [
                df[['rule', 'negative_example_1']].rename(columns={'negative_example_1': 'body'}),
                df[['rule', 'negative_example_2']].rename(columns={'negative_example_2': 'body'}),
            ],
            axis=0
        ).drop_duplicates()
        all_negatives_df['rule_violation'] = 0
        dfs_to_concat_v.append(all_negatives_df)
    
    
    if ('body' in df.columns) and ('rule_violation' in df.columns):
        all_bodies_df = df[['rule', 'body', 'rule_violation']].copy()
        dfs_to_concat_v.append(all_bodies_df)
        
        
    if len(dfs_to_concat_v) > 1:
        all_examples_df = pd.concat(
            dfs_to_concat_v,
            axis=0
        ).drop_duplicates()
        
    elif len(dfs_to_concat_v) == 1:
        all_examples_df = dfs_to_concat_v[0].drop_duplicates()
        
    else:
        raise Exception('No data found')
        
    return all_examples_df.reset_index(drop=True)


class JigsawDatasetCLF():
    def __init__(
        self,
        df,
        tokenizer,
        cleaner=None,
        flip_rules=False,
        all_negatives_to_cls0=True,
        all_positives_to_cls1=False,
        prompt_template="{body}",
    ):
        self.df = df.copy()
        self.tokenizer = tokenizer
        self.cleaner = cleaner
        self.flip_rules = flip_rules
        self.all_negatives_to_cls0 = all_negatives_to_cls0
        self.all_positives_to_cls1 = all_positives_to_cls1
        self.prompt_template = prompt_template
        
        if self.flip_rules:
            self.df['rule'] = self.df['rule'].apply(lambda x: flip_rule(x))
        
        if self.cleaner:
            self.df['rule'] = self.df['rule'].apply(lambda x: self.cleaner(x))
            self.df['body'] = self.df['body'].apply(lambda x: self.cleaner(x))
            
        self.all_rules_v = sorted(self.df['rule'].unique())
        
        if 'rule_violation' in self.df.columns:
            if flip_rules:
                self.df['rule_violation'] = 1 - self.df['rule_violation']
            
            self.positive_examples_d = self.df[['rule', 'body']][self.df['rule_violation'] == 1].groupby('rule').agg(list).to_dict()['body']
            self.negative_examples_d = self.df[['rule', 'body']][self.df['rule_violation'] == 0].groupby('rule').agg(list).to_dict()['body']
            
            if self.all_negatives_to_cls0 and not self.all_positives_to_cls1:
                self.df['label'] = self.df['rule'].apply(lambda r: self.all_rules_v.index(r) + 1)
                self.df.loc[self.df['rule_violation'] == 0, 'label'] = 0
            
            elif self.all_negatives_to_cls0 and self.all_positives_to_cls1:
                self.df['label'] = self.df['rule_violation']
            
            elif not self.all_negatives_to_cls0 and self.all_positives_to_cls1:
                raise NotImplementedError('Not Supported')
            
            elif not self.all_negatives_to_cls0 and not self.all_positives_to_cls1:
                self.df['label'] = self.df['rule'].apply(lambda r: self.all_rules_v.index(r))
                self.df.loc[self.df['rule_violation'] == 0, 'label'] = self.df.loc[self.df['rule_violation'] == 0, 'label'] + len(self.all_rules_v)
                
        return None
    
    def get_num_classes(self):
        if self.all_negatives_to_cls0 and not self.all_positives_to_cls1:
            return 1 + len(self.all_rules_v)
        
        elif self.all_negatives_to_cls0 and self.all_positives_to_cls1:
            return 2
        
        elif not self.all_negatives_to_cls0 and self.all_positives_to_cls1:
                raise NotImplementedError('Not Supported')
        
        elif not self.all_negatives_to_cls0 and not self.all_positives_to_cls1:
            return len(self.all_rules_v) * 2
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        
        text = self.prompt_template.format(**row)
        
        sample_d = self.tokenizer(text)
        #sample_d = self.tokenizer(row['body'])
        
        if 'label' in row:
            sample_d['labels'] =  row['label']
            
        return sample_d
    
    def __len__(self):
        return len(self.df)

    def get_all_bodies(self):
        return self.df['body'].unique().tolist()
    
    def get_all_rules(self):
        return self.all_rules_v
    


def load_trn_fit_df():
    # Train dataset and val dataset
    df_public = pd.read_csv("../input/jigsaw-agile-community-rules/train.csv").sample(frac=1.0, random_state=DS_SEED).reset_index(drop=True)
    
    if PUBLIC_DS_TRN_PROP == 1:
        df_public_trn = df_public.reset_index(drop=True).copy()
        
    elif PUBLIC_DS_TRN_PROP == 0:
        df_public_trn = df_public.iloc[:0]
        
    else:
        df_public_trn = train_test_split(
            df_public,
            train_size=PUBLIC_DS_TRN_PROP,
            random_state=DS_SEED,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    
    if PUBLIC_DS_FIT_PROP == 1:
        df_public_fit = df_public.reset_index(drop=True).copy()
    
    elif PUBLIC_DS_TRN_PROP == 0:
        df_public_fit = df_public.iloc[:0]
        
    else:
        df_public_fit = train_test_split(
            df_public,
            train_size=PUBLIC_DS_FIT_PROP,
            random_state=DS_SEED+1,
            shuffle=True,
            stratify=df_public.rule,
        )[0].reset_index(drop=True)
    
    df_private_tst = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")
    #df_private_tst = pd.read_csv("../input/syn-datasets/syn_train_v2.csv")
    
    df_trn = pd.concat(
        [
            create_clf_dataset(df_public_trn),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body']).reset_index(drop=True).copy()
    
    df_fit = pd.concat(
        [
            create_clf_dataset(df_public_fit),
            create_clf_dataset(df_private_tst),
        ], 
        axis=0
    ).drop_duplicates(subset=['rule', 'body','rule_violation']).reset_index(drop=True).copy()

    return df_trn, df_fit


if __name__ == '__main__':
    assert os.path.exists(MODEL_NAME)
    df_trn, df_fit = load_trn_fit_df()

    print()
    print('------ CFG ----------')
    print(f'{MODEL_NAME=}')

    print()
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')
    print()
    print(f'{PUBLIC_DS_TRN_PROP=}')
    print(f'{PUBLIC_DS_FIT_PROP=}')
    print()
    print(f'{len(df_trn)=}')
    print(f'{len(df_fit)=}')
    print()

    if False:
        import pandas as pd
        import sys, os
    
        x = pd.read_csv("../input/jigsaw-agile-community-rules/test.csv")
        if len(x) <= 10:
            os.system('cp ../input/jigsaw-agile-community-rules/sample_submission.csv ./submission.csv')

## train_clf.py

In [ ]:
%%writefile train_clf.py

import math
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from settings_clf import *


def mean_pooling(last_hidden_state, attention_mask):
    token_embeddings = last_hidden_state  # [B, L, H]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


def last_token_pooling(last_hidden_state, attention_mask):
    last_token_indices = attention_mask.sum(dim=1) - 1  # [batch]
    batch_size = last_hidden_state.size(0)
    hidden_dim = last_hidden_state.size(-1)

    # Obtener embeddings del último token válido
    last_hidden = last_hidden_state[torch.arange(batch_size), last_token_indices]

    return last_hidden  # [batch, hidden_dim]

def first_token_pooling(last_hidden_state, attention_mask):
    last_hidden = last_hidden_state[:,0]
    return last_hidden  # [batch, hidden_dim]


class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        # normalizar
        embeddings = embeddings.to(torch.float32)
        weights = self.weight.to(torch.float32)
        
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)

        cosine = F.linear(embeddings, weights)
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.m)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        output *= self.s
        logits = self.s * cosine
        loss = F.cross_entropy(output, labels)
        return loss, logits
    
    def project(self, embeddings):
        # weights norm
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, weights)
        logits = self.s * cosine
        return logits

class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5, eps=1e-7, reduce_cls0_weight=True):
        super().__init__()
        self.s = s
        self.m = m
        self.eps = eps
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        if reduce_cls0_weight:
            w = torch.ones(out_features)
            w[0] = 1.0 / (out_features - 1)
            self.register_buffer("class_weights", w)
        else:
            self.class_weights = None

    def forward(self, embeddings, labels):
        # 🔹 estabilidad: trabajar siempre en float32
        embeddings = embeddings.to(torch.float32)
        W = self.weight.to(torch.float32)

        embeddings = F.normalize(embeddings, p=2, dim=1)
        W = F.normalize(W, p=2, dim=1)

        cosine = F.linear(embeddings, W)
        cosine = torch.clamp(cosine, -1.0 + self.eps, 1.0 - self.eps)

        sine = torch.sqrt(torch.clamp(1.0 - cosine ** 2, min=0.0))

        phi = cosine * self.cos_m - sine * self.sin_m

        one_hot = torch.zeros_like(cosine, device=embeddings.device)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        logits *= self.s

        loss = F.cross_entropy(
            logits,
            labels,
            weight=self.class_weights.to(embeddings.device) if self.class_weights is not None else None,
        )
        return loss, logits

    def project(self, embeddings):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        W = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, W)
        return self.s * cosine

class XELoss(nn.Module):
    def __init__(self, in_features, out_features, reduce_cls0_weight=True):
        super().__init__()
        
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        if reduce_cls0_weight:
            w = torch.ones(out_features)
            w[0] = 1.0 / (out_features - 1)
            self.register_buffer("class_weights", w)
        else:
            self.class_weights = None
            
    def project(self, embeddings):
        embeddings = embeddings.to(torch.float32)
        W = self.weight.to(torch.float32)
        logits = F.linear(embeddings, W)
        return logits

    def forward(self, embeddings, labels):
        logits = self.project(embeddings)

        loss = F.cross_entropy(
            logits,
            labels,
            weight=self.class_weights.to(embeddings.device) if self.class_weights is not None else None,
        )
        return loss, logits


class EmbeddingModel(nn.Module):
    def __init__(self, model, loss_fn, pooling_type):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn
        
        if pooling_type == 'mean':
            self.pooling_fn = mean_pooling
            
        elif pooling_type == 'last_token':
            self.pooling_fn = last_token_pooling
            
        elif pooling_type == 'first_token':
            self.pooling_fn = first_token_pooling
            
        else:
            raise NotImplementedError(f'{pooling_type=} ???')
        
        return None

    def forward(self, input_ids, attention_mask, labels=None, **kw_args):
        call_args_d = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
        }
        
        if 'token_type_ids' in kw_args.keys():
            call_args_d['token_type_ids'] = kw_args['token_type_ids']
            
        outputs = self.model(**call_args_d)
            
        embeddings = self.pooling_fn(outputs.last_hidden_state, attention_mask)
        
        if labels is not None:
            #print(f'{type(embeddings)=} {embeddings.shape=} {embeddings.dtype=} {embeddings=}')
            #print(f'{type(labels)=} {labels.shape=} {labels.dtype=} {labels=}')
            loss, logits = self.loss_fn(embeddings, labels)
            #print(f'{type(loss)=} {loss=}')
            #print(f'{type(logits)=} {logits=}')
            return {"loss": loss, "embeddings": embeddings, "logits": logits}
        
        logits = self.loss_fn.project(embeddings)
        return {"embeddings": embeddings, "logits": logits}
    
    def gradient_checkpointing_enable(self, **kwargs):
        if hasattr(self.model, "gradient_checkpointing_enable"):
            self.model.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        if hasattr(self.model, "gradient_checkpointing_disable"):
            self.model.gradient_checkpointing_disable()
    
    def save_model(self, outputs_dir):
        self.model.save_pretrained(outputs_dir)
        
        save_obj(
            self.loss_fn.state_dict(),
            os.path.join(outputs_dir, 'loss_weights.pickle'),
        )
        
        return None
    
    def print_trainable_parameters(self):
        n_total_trainable = 0
        n_total = 0
        for n, p in self.named_parameters():
            if p.requires_grad:
                #print(f'[GRAD] {n:70s} {list(p.shape)}')
                n_total_trainable += p.numel()
                n_total += p.numel()
            else:
                #print(f'[    ] {n:70s} {list(p.shape)}')
                n_total += p.numel()
            
        print(f'Num trainable params: {n_total_trainable} / {n_total}  ({n_total_trainable/n_total:0.02%})')

        return None 
    
    
    def encode(
        self, 
        ds,
        data_collator,
        batch_size: int = 32, 
        normalize: bool = True,
        output_numpy: bool = False,
        verbose: bool = True,
    ):

        self.model.eval()

        embeddings_v = []
        logits_v = []
        with torch.no_grad():
            itt = DataLoader(ds, batch_size=batch_size, collate_fn=data_collator)
            if verbose:
                itt = tqdm(itt)
                
            for data in itt:
                data = data.to(self.model.device)
                
                data['labels'] = None
                
                with torch.autocast(device_type="cuda", dtype=None):
                    outputs = self(**data)
                    
                embeddings = outputs['embeddings']
                logits = outputs['logits']
                
                if normalize:
                    embeddings = F.normalize(embeddings, p=2, dim=1)

                embeddings_v.append(embeddings.cpu())
                logits_v.append(logits.cpu())

            embeddings_v = torch.cat(embeddings_v, dim=0)
            logits_v = torch.cat(logits_v, dim=0)
        
            if output_numpy:
                embeddings_v = embeddings_v.numpy()
                logits_v = logits_v.numpy()
            
        return embeddings_v, logits_v
    
    
def load_base_model(model_name, add_lora=True, init_lora=None, is_trainable=True):
    from peft import get_peft_model, LoraConfig, TaskType, PeftModel
    from transformers import AutoModel, AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModel.from_pretrained(model_name)
    
    torch.manual_seed(SEED)
    if add_lora:
        if init_lora is None:
            # Configure LoRA parameters
            lora_config = LoraConfig(
                r=LORA_RANK,  # Rank of the update matrices
                lora_alpha=LORA_ALPHA,  # Alpha parameter for LoRA scaling
                lora_dropout=0.05,  # Dropout probability for LoRA layers
                task_type=TaskType.FEATURE_EXTRACTION,
                bias='none',  # Don't train bias terms
                # Target the attention and MLP modules of the transformer
                target_modules=[
                    "q_proj",
                    "k_proj",
                    "v_proj",
                    "o_proj",
                    "gate_proj",
                    "up_proj",
                    "down_proj",
                ]
            )

            model = get_peft_model(base_model, lora_config)
        
        
        else:
            model = PeftModel.from_pretrained(
                base_model,
                init_lora,
                is_trainable=is_trainable,
            )
            
    else:
        model = base_model
        
    return model, tokenizer


def load_embedding_model(base_model, init_lora, num_classes=2, load_loss_weights=True, is_trainable=True):
    from transformers import DataCollatorWithPadding
    
    loss_type = LOSS_TYPE
    print(f'Loading: {base_model=} for {loss_type=}')
    model, tokenizer = load_base_model(base_model, add_lora=USE_LORA, init_lora=init_lora, is_trainable=is_trainable)
    
    if load_loss_weights:
        loss_weights_path = os.path.join(init_lora or base_model, 'loss_weights.pickle')
        
        if not os.path.exists(loss_weights_path):
            print(f'- 🚨 Impossible to load weights: {loss_weights_path=} does not exist.')
            load_loss_weights = False
            
        else:
            print(f'- Loading loss weights: {loss_weights_path=}')
            loss_sd = load_obj(loss_weights_path)
            
            if loss_sd['weight'].shape[0] != num_classes:
                print(f"- 🚨 Setting {num_classes=} to {loss_sd['weight'].shape[0]}")
                num_classes = loss_sd['weight'].shape[0]
            
    torch.manual_seed(SEED+2)
    
    if loss_type == 'AF':
        # Initial Model
        loss_fn = ArcFaceLoss(
            in_features=model.config.hidden_size,
            out_features=num_classes,
            s=ARCFACE_SCALE,
            m=ARCFACE_MARGIN,
            reduce_cls0_weight=REDUCE_CLS0_WEIGHT,
        )
        
    elif loss_type == 'XE':
        loss_fn = XELoss(
            in_features=model.config.hidden_size,
            out_features=num_classes,
            reduce_cls0_weight=REDUCE_CLS0_WEIGHT,
        )
        
    else:
        raise NotImplementedError(f'{loss_type=} ???')
        
    if load_loss_weights:
        loss_fn.load_state_dict(loss_sd)
    
    embeddings_model = EmbeddingModel(
        model,
        loss_fn,
        pooling_type=POOLING_TYPE,
    )
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    
    return embeddings_model, tokenizer, data_collator


def run_inference_on_device(clf_df, base_model, init_lora):
    embeddings_model, tokenizer, data_collator = load_embedding_model(
        base_model,
        init_lora,
        num_classes=2,
        load_loss_weights=True,
        is_trainable=False
    )
    
    embeddings_model.to('cuda:0')
    
    ds = JigsawDatasetCLF(
        clf_df,
        tokenizer=tokenizer,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
        all_negatives_to_cls0=ALL_NEGATIVES_TO_CLS0,
        all_positives_to_cls1=ALL_POSITIVES_TO_CLS1,
        prompt_template=PROMPT_TEMPLATE,
    )
     
    embeddings_v, logits_v = embeddings_model.encode(
        ds,
        data_collator,
        batch_size=EVAL_BS, 
        normalize=True,
        output_numpy=False,
        verbose=True,
    )
    proc_df = ds.df.copy()
    return proc_df, embeddings_v, logits_v
    

def worker(device_id, clf_df, base_model, init_lora, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] 🔥 Running on GPU {device_id}, {len(clf_df)=} {init_lora=}")
    
    proc_df, embeddings_v, logits_v = run_inference_on_device(clf_df, base_model, init_lora)

    return_dict[f'proc_df_slice{device_id}'] = proc_df
    return_dict[f'embeddings_v_slice{device_id}'] = embeddings_v
    return_dict[f'logits_v_slice{device_id}'] = logits_v
    print(f"[Worker {device_id}] ✅ Inference OK!")
    
    
def eval_model_embeddings(clf_df, base_model, init_lora):
    import multiprocessing as mp
    
    print('😱 Starting Embeddings Inference:')

    rows_mid = len(clf_df) // 2

    manager = mp.Manager()
    return_dict = manager.dict()
    
    p0 = mp.Process(
        target=worker,
        args=(
            0, # Device
            clf_df.iloc[:rows_mid],
            base_model,
            init_lora,
            return_dict,
        )
    )
    p1 = mp.Process(
        target=worker,
        args=(
            1, # Device
            clf_df.iloc[rows_mid:],
            base_model,
            init_lora,
            return_dict,
        )
    )
    p0.start()
    p1.start()
    p0.join()
    p1.join()
    
    embeddings_v = torch.concat(
        [
            return_dict['embeddings_v_slice0'],
            return_dict['embeddings_v_slice1'],
        ]
    )
    logits_v = torch.concat(
        [
            return_dict['logits_v_slice0'],
            return_dict['logits_v_slice1'],
        ]
    )
    
    proc_df = pd.concat(
        [
            return_dict['proc_df_slice0'],
            return_dict['proc_df_slice1'],
        ]
    )
    
    return proc_df, embeddings_v, logits_v


    
def main():
    df_trn, df_fit = load_trn_fit_df()
    
    ds_trn = JigsawDatasetCLF(
        create_clf_dataset(df_trn),
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
        all_negatives_to_cls0=ALL_NEGATIVES_TO_CLS0,
        all_positives_to_cls1=ALL_POSITIVES_TO_CLS1,
        prompt_template=PROMPT_TEMPLATE,
    )
    
    print(f'{len(ds_trn)=}')

    if CALC_INIT_LOSS_WEIGHTS:
        print('Initial trn embeddings eval')
        #trn_proc_df, trn_embeddings_v, logits_v = eval_model_embeddings(ds_trn.df[ds_trn.df.label!=0].copy(), base_model=MODEL_NAME, init_lora=INIT_LORA)
        trn_proc_df, trn_embeddings_v, logits_v = eval_model_embeddings(ds_trn.df.copy(), base_model=MODEL_NAME, init_lora=INIT_LORA)



    from transformers import Trainer, TrainingArguments
    from transformers.utils import is_torch_bf16_gpu_available

    NUM_CLASSES = ds_trn.get_num_classes()

    embeddings_model, tokenizer, data_collator = load_embedding_model(
        base_model=MODEL_NAME,
        init_lora=INIT_LORA,
        num_classes=NUM_CLASSES,
        load_loss_weights=False,
        is_trainable=True,
    )
    ds_trn.tokenizer = tokenizer
    ds_val = None
        
    if CALC_INIT_LOSS_WEIGHTS:
        for label in sorted(trn_proc_df['label'].unique().tolist()):
            print(f'- Setting centroid: {label=}')
            f = (trn_proc_df['label'] == label).values
            w = F.normalize(trn_embeddings_v[f].mean(axis=0), dim=0, p=2)
            w = w.type(embeddings_model.loss_fn.weight.data.dtype).to(embeddings_model.loss_fn.weight.data.device)
            
            if ALL_NEGATIVES_TO_CLS0 and (label == 0) and (LOSS_TYPE == 'AF'):
                embeddings_model.loss_fn.weight.data[label,:] = 0.5 * embeddings_model.loss_fn.weight.data[label,:] + w
                
            else:
                embeddings_model.loss_fn.weight.data[label,:] = w
    
    
    # Calculate max_steps for small datasets
    NUM_DEVICES = torch.cuda.device_count()
    
    if os.path.exists(MODEL_OUTPUT_PATH):
        os.system(f'rm -r {MODEL_OUTPUT_PATH}')
    
    embeddings_model.print_trainable_parameters()
                
    LOGGING_STRATEGY = ['epoch', 'steps'][1]
    if ds_val is None:
        SAVE_STRATEGY = 'no'
        EVAL_STRATEGY = 'no'
    else:
        SAVE_STRATEGY = LOGGING_STRATEGY
        EVAL_STRATEGY = LOGGING_STRATEGY


    LOGGING_STEPS = len(ds_trn) // (TRAIN_BS * GRAD_ACC_NUM * N_EVALS_PER_EPOCH * torch.cuda.device_count())
    SAVE_TOTAL_LIMIT = N_EVALS_PER_EPOCH + 1

    SAVE_STEPS = LOGGING_STEPS

    print('Training ...')
    print()
    print('------ CFG----------')
    print(f'{NUM_DEVICES=}')
    print(f'{NUM_CLASSES=}')
    print(f'{EPOCHS=}')
    print(f'{EVAL_STRATEGY=}')
    print(f'{LOGGING_STEPS=}')
    print(f'{SAVE_STEPS=}')
    print(f'{SAVE_TOTAL_LIMIT=}')
    print(f'{TRAIN_BS=}')
    print(f'{EVAL_BS=}')
    print(f'{GRAD_ACC_NUM=}')

    print(f'{len(ds_trn)=}')
    if ds_val is None:
        print(f'{ds_val=}')
    else:
        print(f'{len(ds_val)=}')
    print('----------------')
    print()

    # Set up training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        
        num_train_epochs=EPOCHS,
        optim="paged_adamw_8bit",  # 8-bit optimizer for memory efficiency
        #optim="adamw_torch",  # 8-bit optimizer for memory efficiency
        lr_scheduler_type="linear",
        warmup_ratio=0.1,  # Warm up learning rate over 10% of steps
        learning_rate=LR,
        weight_decay=0.01,
        #max_grad_norm=1.0,
        
        # Use BF16 if available, otherwise FP16
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),

        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACC_NUM,
        gradient_checkpointing=True,  # Save memory with gradient checkpointing
        gradient_checkpointing_kwargs={"use_reentrant": False},
        group_by_length=False,
        seed=SEED,
        remove_unused_columns=False,  # Keep all columns in the dataset
        
        report_to="tensorboard",
        metric_for_best_model="AUC",
        greater_is_better=True,

        load_best_model_at_end=(ds_val is not None),
        save_total_limit=SAVE_TOTAL_LIMIT,

        logging_strategy=LOGGING_STRATEGY,
        eval_strategy=EVAL_STRATEGY,
        save_strategy=SAVE_STRATEGY,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
    )

    # Initialize trainer
    trainer = Trainer(
        embeddings_model,
        args=training_args,
        train_dataset=ds_trn,
        eval_dataset=ds_val,
        data_collator=data_collator,
    )
    
    trainer.train()
    
    print(f"Saving fine-tuned model to {MODEL_OUTPUT_PATH}...")
    embeddings_model.save_model(MODEL_OUTPUT_PATH)
    if not USE_LORA:
        tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
        
    return None


if __name__ == '__main__':
    main()

## embeddings_inference_clf.py

In [ ]:
%%writefile embeddings_inference_clf.py

import os
import random
import argparse
import pandas as pd
import numpy as np
import random

import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

from settings_clf import *
from train_clf import eval_model_embeddings

def fit_gbm_models(X_trn, y_trn, X_tst, n_splits=4, seed=2434, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)

    print(f'LGBM: {X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'LGBM: fitting fold: {i_fold}')

        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)

        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1
        }
        if seed:
            params['seed'] = seed + i_fold
            
        callbacks_v = [lgb.early_stopping(stopping_rounds=50)]

        if verbose:
            callbacks_v.append(lgb.log_evaluation(period=50))
            
        # Entrenar
        model = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=callbacks_v
        )

        # Predicciones
        y_fold_pred = model.predict(X_tst, num_iteration=model.best_iteration)
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)

    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v

def fit_xgb_models(X_trn, y_trn, X_tst, n_splits=4, seed=234, verbose=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=4342)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        
    print(f'XGB: {X_trn.shape=} {y_trn.shape=} {X_tst.shape=}')
    
    fold_models_v = []
    test_fold_predictions_v = []
    for i_fold, (train_idx, valid_idx) in enumerate(skf.split(X_trn, y_trn)):
        print(f'XGB: fitting fold: {i_fold}')
        
        X_train, X_valid = X_trn[train_idx], X_trn[valid_idx]
        y_train, y_valid = y_trn[train_idx], y_trn[valid_idx]
        
        model = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.1,
            max_depth=10,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            early_stopping_rounds=25, 
            metric='auc',
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=verbose
        )
        # Predicciones
        y_fold_pred = model.predict(X_tst, iteration_range=(0, model.best_iteration + 1))
        test_fold_predictions_v.append(y_fold_pred)
        fold_models_v.append(model)
        
    tst_final_pred = np.mean(test_fold_predictions_v, axis=0)
    
    tst_final_pred = np.concatenate(
        [
            1 - tst_final_pred[:, None], 
            tst_final_pred[:, None]
        ],
        axis=1
    )
    
    return tst_final_pred, test_fold_predictions_v, fold_models_v


def fit_and_predict_embeddings(fit_samples_df, tst_samples_df):
    all_tst_preds_v = np.zeros( (len(tst_samples_df), 2) )
    n_clf = 0
    for i_rule, rule in enumerate(tst_samples_df.rule.unique()):
        f_fit = (fit_samples_df['rule'] == rule)
        f_pred = (tst_samples_df.rule.values == rule)

        if 'RB_LG' in OUTPUT_PREDICTOR_CFG:
            print(f'Fitting: LogisticRegression DS[rule={i_rule}]')
            clf = LogisticRegression(max_iter=1000, C=1)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1

        if 'RB_SVC' in OUTPUT_PREDICTOR_CFG:
            print(f'Fitting: SVC rule: DS[rule={i_rule}]')
            clf = SVC(kernel="linear", C=2, probability=True, random_state=42)
            clf.fit(
                np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                fit_samples_df[f_fit]['rule_violation'].values,
            )
            all_tst_preds_v[f_pred] += clf.predict_proba(np.vstack(tst_samples_df[f_pred]['embeddings'].values))
            if i_rule == 0:
                n_clf += 1


        if 'RB_LGBM' in OUTPUT_PREDICTOR_CFG:
            print(f'🟠 Fitting: LGBM rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_gbm_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1

        if 'RB_XGB' in OUTPUT_PREDICTOR_CFG:
            print(f'🟢 Fitting: XGB rule: DS[rule={i_rule}]')
            all_tst_preds_v[f_pred] += fit_xgb_models(
                X_trn=np.vstack(fit_samples_df[f_fit]['embeddings'].values),
                y_trn=fit_samples_df[f_fit]['rule_violation'].values,
                X_tst=np.vstack(tst_samples_df[f_pred]['embeddings'].values),
                n_splits=4
            )[0]
            if i_rule == 0:
                n_clf += 1
            
    if 'FDS_LG' in OUTPUT_PREDICTOR_CFG:
        print('Fitting: LogisticRegression FullDS')
        clf = LogisticRegression(max_iter=1000, C=2)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1

    if 'FDS_SVC' in OUTPUT_PREDICTOR_CFG:
        print('Fitting: SVC FullDS')
        clf =SVC(kernel="linear", C=2, probability=True, random_state=42)
        clf.fit(
            np.vstack(fit_samples_df['embeddings'].values),
            fit_samples_df['rule_violation'].values,
        )
        all_tst_preds_v += clf.predict_proba(np.vstack(tst_samples_df['embeddings'].values))
        n_clf += 1


    if 'FDS_LGBM' in OUTPUT_PREDICTOR_CFG:
        print('🟠 Fitting: LGBM FullDS')
        all_tst_preds_v += fit_gbm_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=8
        )[0]
        n_clf += 1
        
        
    if 'FDS_XGB' in OUTPUT_PREDICTOR_CFG:
        print('🟢 Fitting: XGB FullDS')
        all_tst_preds_v += fit_xgb_models(
            X_trn=np.vstack(fit_samples_df['embeddings'].values),
            y_trn=fit_samples_df['rule_violation'].values,
            X_tst=np.vstack(tst_samples_df['embeddings'].values),
            n_splits=4
        )[0]
        n_clf += 1

    all_tst_preds_v = all_tst_preds_v / n_clf

    return all_tst_preds_v


def main():
    parser = argparse.ArgumentParser(description="CLF model inference")
    parser.add_argument("--base-model-path", default=MODEL_NAME if USE_LORA else MODEL_OUTPUT_PATH, type=str)
    parser.add_argument("--lora-model-path", default=MODEL_OUTPUT_PATH if USE_LORA else None, type=str)
    parser.add_argument("--inference-dataset", default="../input/jigsaw-agile-community-rules/test.csv", type=str)
    parser.add_argument("--output-csv", type=str, default='./submission.csv')
    args = parser.parse_args()

    base_model_path = args.base_model_path
    lora_model_path = args.lora_model_path
    inference_dataset = args.inference_dataset
    output_csv = args.output_csv

    print('Inference settings:')
    print(f'{base_model_path=}')
    print(f'{lora_model_path=}')
    print(f'{inference_dataset=}')
    print(f'{output_csv=}')
    print()
    
    df_tst = pd.read_csv(inference_dataset)
    
    if FLIP_RULES:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: flip_rule(x))
    
    if USE_CLEANER:
        df_tst['rule'] = df_tst['rule'].apply(lambda x: cleaner(x))
        df_tst['body'] = df_tst['body'].apply(lambda x: cleaner(x))
        
    _, df_fit = load_trn_fit_df()
    
    ds_fit = JigsawDatasetCLF(
        create_clf_dataset(df_fit),
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
        all_negatives_to_cls0=ALL_NEGATIVES_TO_CLS0,
        all_positives_to_cls1=ALL_POSITIVES_TO_CLS1,
        prompt_template=PROMPT_TEMPLATE,
    )
    
    ds_tst = JigsawDatasetCLF(
        df_tst,
        tokenizer=None,
        cleaner=cleaner if USE_CLEANER else None,
        flip_rules=FLIP_RULES,
        all_negatives_to_cls0=ALL_NEGATIVES_TO_CLS0,
        all_positives_to_cls1=ALL_POSITIVES_TO_CLS1,
        prompt_template=PROMPT_TEMPLATE,
    )
    
    if TRAIN_OUTPUT_PREDICTOR:
        embeddings_df = pd.concat(
            [
                ds_fit.df[['rule', 'body']],
                ds_tst.df[['rule', 'body']],
            ],
            axis=0
        ).drop_duplicates().reset_index(drop=True)
        
    else:
        embeddings_df = ds_tst.df[['rule', 'body']].drop_duplicates().reset_index(drop=True)
    
    proc_df, embeddings_v, logits_v = eval_model_embeddings(embeddings_df, base_model=base_model_path, init_lora=lora_model_path)
    
    if FLIP_RULES:
        probs_v = logits_v.softmax(-1).numpy()[:,0]
    else:
        probs_v = logits_v.softmax(-1).numpy()[:,1]
    
    all_texts2embed_d = {}
    for rule, body, embedding in zip(proc_df.rule.values, proc_df.body.values, embeddings_v):
        all_texts2embed_d[(rule, body)] = embedding
        
    all_texts2probs_d = {}
    for rule, body, probs in zip(proc_df.rule.values, proc_df.body.values, probs_v):
        all_texts2probs_d[(rule, body)] = probs

    
    df_tst['embeddings'] = df_tst[['rule', 'body']].apply(lambda row: all_texts2embed_d[(row.rule, row.body)], axis=1)
    df_tst['probs'] = df_tst[['rule', 'body']].apply(lambda row: all_texts2probs_d[(row.rule, row.body)], axis=1)
    

    if TRAIN_OUTPUT_PREDICTOR:
        ds_fit.df['embeddings'] = ds_fit.df[['rule', 'body']].apply(lambda row: all_texts2embed_d[(row.rule, row.body)], axis=1)
        ds_fit.df['probs'] = ds_fit.df[['rule', 'body']].apply(lambda row: all_texts2probs_d[(row.rule, row.body)], axis=1)
        
        print('☢️ Fitting embeddings CLFs.')
        all_tst_preds_v = fit_and_predict_embeddings(ds_fit.df, df_tst)
        
        if FLIP_RULES:
            df_tst['rule_violation'] = all_tst_preds_v[:, 0] # labels are flipped
        else:
            df_tst['rule_violation'] = all_tst_preds_v[:, 1]
    
    
        if ENS_PROBS_TO_PRED:
            print('⚠️ Ensembling CLF PROBS')
            df_tst['rule_violation'] = 0.5 * (df_tst['rule_violation'] + df_tst['probs'])
    else:
        df_tst['rule_violation'] = df_tst['probs']
    
    
    if os.path.exists('../input/jigsaw-agile-community-rules/test_gt.csv'):
        from sklearn.metrics import accuracy_score, roc_auc_score
        df_tst_gt = pd.read_csv('../input/jigsaw-agile-community-rules/test_gt.csv')
        AUC = roc_auc_score(y_true=df_tst_gt.rule_violation.values, y_score=df_tst.rule_violation.values)
        ACC = accuracy_score(y_true=df_tst_gt.rule_violation.values, y_pred=df_tst.rule_violation.values > 0.5)

        print('Test GT Found, computing Final score:')
        print(f'-Test {AUC=:0.4f}')
        print(f'-Test {ACC=:0.4f}')

        if False:
            save_obj(
                {
                    'all_texts2embed_d': all_texts2embed_d,
                    'df_tst': df_tst,
                    'ds_fit': ds_fit,
                },
                './embeddings.pickle'
            )

    df_tst[['row_id', 'rule_violation']].to_csv(output_csv, index=False)
    print(f"✅ Saved: {output_csv}")

    return None

if __name__ == "__main__":
    main()

## constants_clf.py

In [ ]:
%%writefile constants_clf.py

LORA_RANK = 16
LORA_ALPHA = 16

TEST = 46
DS_SEED = 42 + TEST
SEED = 38209
N_EVALS_PER_EPOCH = 5

LR = 1e-4

PUBLIC_DS_TRN_PROP = 1.00
PUBLIC_DS_FIT_PROP = 1.00

TRAIN_BS = 32
EVAL_BS = 64
GRAD_ACC_NUM = 1

OUTPUT_DIR = f"./clf_embed_output"
MODEL_OUTPUT_PATH = f"{OUTPUT_DIR}/best_model"

if True: #-Test AUC=0.8605 -Test ACC=0.7553
    MODEL_NAME = "../input/huggingfacedebertav3variants/deberta-v3-small"
    USE_LORA = False
    INIT_LORA = None
    LOSS_TYPE = ['AF', 'XE'][1]

    EPOCHS = 4 # Training epochs
    PUBLIC_DS_TRN_PROP = 0.05
    PUBLIC_DS_FIT_PROP = 1.00
    
    MAX_SEQ_LENGTH = 250
    DO_LOWER_CASE = False
    USE_CLEANER = True

    POOLING_TYPE = ['mean', 'last_token', 'first_token'][-1]
    CALC_INIT_LOSS_WEIGHTS = False
    ARCFACE_MARGIN = 0.3
    ARCFACE_SCALE = 32.0

    FLIP_RULES = False
    ALL_NEGATIVES_TO_CLS0 = True
    ALL_POSITIVES_TO_CLS1 = True
    REDUCE_CLS0_WEIGHT = False
    
    PROMPT_TEMPLATE = '{rule}[SEP]{body}'
    TRAIN_OUTPUT_PREDICTOR = True
    ENS_PROBS_TO_PRED = True
    OUTPUT_PREDICTOR_CFG = [
        #'RB_LG',
        #'RB_SVC',
        'RB_LGBM',
        #'RB_XGB',
        
        #'FDS_LG',
        #'FDS_SVC',
        'FDS_LGBM',
        #'FDS_XGB',
    ]


## RUN

In [ ]:
!python settings_clf.py

In [ ]:
!python train_clf.py

In [ ]:
!python embeddings_inference_clf.py --inference-dataset /kaggle/input/jigsaw-agile-community-rules/test.csv --output-csv /kaggle/working/submission_deberta_ttt.csv

# Ensemble

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
subs_to_ensemble_v = [
    'submission_triplet_ttt_RF.csv',
    'submission_triplet_ttt_noRF.csv',
    'submission_arcface_ttt.csv',
    'submission_deberta_ttt.csv',
]

df_to_ensemble_v = [
    pd.read_csv(sub_path) if isinstance(sub_path, str) else sub_path[1]
    for sub_path in subs_to_ensemble_v
]

scores_to_ensemble_v = [
    df['rule_violation']
    for df in df_to_ensemble_v
]

w_v = [1.0, 1.0, 0.8, 0.8]
assert len(w_v) == len(scores_to_ensemble_v)

ens_score = None
for w, score in zip(w_v, scores_to_ensemble_v):
    if ens_score is None:
        ens_score = w * score
    else:
        ens_score = ens_score + w * score

ens_score = ens_score / sum(w_v)


sub_models_ensemble_df = df_to_ensemble_v[0].copy()
sub_models_ensemble_df['rule_violation'] = ens_score
#submission_df.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
if len(ens_score) == 10:
    from matplotlib import pyplot as plt

    for score, name in zip(scores_to_ensemble_v, subs_to_ensemble_v):
        plt.plot(score, label=str(name))
        
    plt.plot(ens_score, 'k-o', label='ens_score')
    
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
subs_to_ensemble_v = [
    'submission_ttt_WQ25_14b_0.csv',
    'submission_ttt_WQ25_14b_1.csv',

    'submission_ttt_WQ3_14b_0.csv',
    'submission_ttt_WQ3_14b_1.csv',

    'submission_ttt_WQ25_7b_merge.csv',
    'submission_ttt_WQ3_8b_merge.csv',

    #'./submission_ttt_7b_QW2.csv',
    #'./submission_ttt_8b_QW3.csv',
    
    ('sub_models_ensemble_df', sub_models_ensemble_df),
]

df_to_ensemble_v = [
    pd.read_csv(sub_path) if isinstance(sub_path, str) else sub_path[1]
    for sub_path in subs_to_ensemble_v
]

scores_to_ensemble_v = [
    df['rule_violation']
    for df in df_to_ensemble_v
]

w_v = [1.5, 1.5, 1.3, 1.3, 1.0, 0.8, 1.0]
assert len(w_v) == len(scores_to_ensemble_v)

ens_score = None
for w, score in zip(w_v, scores_to_ensemble_v):
    if ens_score is None:
        ens_score = w * score
    else:
        ens_score = ens_score + w * score

ens_score = ens_score / sum(w_v)


submission_df = df_to_ensemble_v[0].copy()
submission_df['rule_violation'] = ens_score
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
import pandas as pd
pd.read_csv('/kaggle/working/submission.csv')

In [ ]:
if len(ens_score) == 10:
    from matplotlib import pyplot as plt

    for score, name in zip(scores_to_ensemble_v, subs_to_ensemble_v):
        plt.plot(score, label=str(name) if isinstance(name, str) else name[0])
        
    plt.plot(ens_score, 'k-o', label='ens_score')
    
    plt.legend()
    plt.grid()
    plt.show()